# AFPHA-DAGSNet -- T4 calibration

Measures on the real hardware what a laptop GPU could only estimate. The important part
is **section 5: the production driver runs for real** on the 20-client partition with
both GPUs and the full test set, so parquet decode, worker startup, compile validation,
resident-VRAM fit, eval sharding and commit I/O are measured rather than assumed.
Section 4 is a synthetic micro-benchmark kept only to compare backends.

Writes `calibration.json` with a suggested `MAX_HOURS` per scenario.
**Expect roughly 30-50 minutes.** No 50-round training happens here.

## 1. Hardware gate

In [ ]:
import json, os, subprocess, sys, time
import torch

# Two Tesla T4s is a precondition, not a preference: the schedule, the resident-data
# budget and the eval sharding all assume it. Fail here rather than 6 hours in.
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
caps  = [torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())]
print("devices:", names, caps)
assert len(names) == 2, f"expected 2 GPUs, got {len(names)}: {names}"
assert all("T4" in n for n in names), f"expected Tesla T4s, got {names}"
assert all(c == (7, 5) for c in caps), f"expected capability (7,5), got {caps}"
print("torch", torch.__version__, "| cuda", torch.version.cuda)
for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  gpu{i}: {free/2**30:.1f} GiB free of {total/2**30:.1f} GiB")
print("host RAM:", subprocess.run(["free","-g"], capture_output=True, text=True).stdout.splitlines()[1])
print("input mounts:", sorted(os.listdir("/kaggle/input")))

In [ ]:
CAL_ROUNDS    = 3      # round 1 pays compile; the rest give the steady-state cost
CAL_MAX_HOURS = 1.5
RUN_DIR = "/kaggle/working"

## 2. Can this session read the `wandb_key` secret?

This decides how the three training runs get launched. A Kaggle API/CLI push has been
observed to create a version with **no** secret attached, which would mean the owner has
to launch each training notebook from the editor with *Save Version > Save & Run All*.
This cell answers the question for the price of one calibration run instead of one
wasted training run. **It never raises** -- a failure here is a result, not an error.

In [ ]:
# wandb is present in most Kaggle images but that is not a guarantee, and this notebook
# cannot proceed without it. Internet is enabled, so install it rather than failing.
try:
    import wandb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb
print("wandb", wandb.__version__)

# Try the Kaggle secret named "wandb_key" first. Only its length is ever printed.
SECRET = {}
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret("wandb_key")
    SECRET = {"available": True, "length": len(key)}
    os.environ["WANDB_API_KEY"] = key
    os.environ["WANDB_SILENT"] = "true"
    # The key goes in as an argument, not only as an environment variable. relogin=True
    # tells wandb to discard whatever credential it already has and authenticate again,
    # and in that path it does NOT read WANDB_API_KEY -- it goes looking for a netrc or a
    # terminal prompt. A Kaggle kernel has neither, so the env-var-only form raises
    # "No API key configured" there while working locally, where ~/.netrc happens to exist.
    SECRET["login"] = bool(wandb.login(key=key, anonymous="never", relogin=True))
    SECRET["entity"] = wandb.Api().default_entity
except Exception as exc:
    SECRET = {"available": False, "error": f"{type(exc).__name__}: {exc}"}
print(json.dumps(SECRET, indent=2))

print()
print("launch path:",
      "secret visible -> either path launches training"
      if SECRET.get("login") else
      "no secret -> the training notebooks fall back to their inline key, which also "
      "makes an API/CLI push self-sufficient")

## 3. Training modules

In [ ]:
MODULES = {
 "dagsnet.py": "\"\"\"DAGSNet \u2014 Khan et al. 2025 \u00a74.10, Eq. (38)-(48). 395,024 learnable parameters.\nIn: (B, 66) z-scored features.  Out: (B, 16) logits (NO softmax).\n\nCopied verbatim from knowledge/architecture.md \u00a76. Do not change CFG: the\nparameter count and state_dict key set are part of the reconstruction contract.\n\"\"\"\nimport torch\nimport torch.nn as nn\n\n\ndef cbr(i, o, k):\n    return nn.Sequential(nn.Conv1d(i, o, k, padding=k // 2, bias=False),\n                         nn.BatchNorm1d(o), nn.ReLU(inplace=True))\n\n\nclass DenseNet1d(nn.Module):\n    def __init__(self, cin, growth, layers):\n        super().__init__()\n        self.blocks = nn.ModuleList([cbr(cin + i * growth, growth, 3) for i in range(layers)])\n        self.out_ch = cin + layers * growth\n\n    def forward(self, x):\n        for b in self.blocks:\n            x = torch.cat([x, b(x)], dim=1)                     # Eq. (38)\n        return x                                                # Eq. (39)\n\n\nclass Inception1d(nn.Module):\n    def __init__(self, cin, c):\n        super().__init__()\n        self.b1 = cbr(cin, c, 1)\n        self.b3 = nn.Sequential(cbr(cin, c, 1), cbr(c, c, 3))\n        self.b5 = nn.Sequential(cbr(cin, c, 1), cbr(c, c, 5))\n        self.bp = nn.Sequential(nn.MaxPool1d(3, 1, 1), cbr(cin, c, 1))\n        self.out_ch = 4 * c\n\n    def forward(self, x):\n        return torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1)\n\n\nclass GoogleNet1d(nn.Module):\n    def __init__(self, cin, modules_n, c=32):\n        super().__init__()\n        mods, ch = [], cin\n        for _ in range(modules_n):\n            m = Inception1d(ch, c); mods.append(m); ch = m.out_ch\n        self.net = nn.Sequential(*mods); self.out_ch = ch\n\n    def forward(self, x):\n        return self.net(x)\n\n\nclass AlexNet1d(nn.Module):\n    def __init__(self, cin, ch=128):\n        super().__init__()\n        self.net = nn.Sequential(\n            cbr(cin, ch, 3), nn.MaxPool1d(2, ceil_mode=True),\n            cbr(ch, ch, 3),  nn.MaxPool1d(2, ceil_mode=True),\n            cbr(ch, ch, 3))\n        self.out_ch = ch\n\n    def forward(self, x):\n        return self.net(x)\n\n\nclass Fire1d(nn.Module):\n    def __init__(self, cin, sq, ex):\n        super().__init__()\n        self.squeeze = cbr(cin, sq, 1)                          # Eq. (46)\n        self.e1 = cbr(sq, ex, 1)\n        self.e3 = cbr(sq, ex, 3)\n        self.out_ch = 2 * ex\n\n    def forward(self, x):\n        s = self.squeeze(x)\n        return torch.cat([self.e1(s), self.e3(s)], dim=1)       # Eq. (45)\n\n\nclass SqueezeNet1d(nn.Module):\n    def __init__(self, cin, modules_n, sq=32, ex=48):\n        super().__init__()\n        mods, ch = [], cin\n        for _ in range(modules_n):\n            m = Fire1d(ch, sq, ex); mods.append(m); ch = m.out_ch\n        self.net = nn.Sequential(*mods); self.out_ch = ch\n\n    def forward(self, x):\n        return self.net(x)\n\n\nclass DAGSNet(nn.Module):\n    def __init__(self, cfg, n_features):\n        super().__init__()\n        self.patch_len = cfg[\"patch_len\"]\n        assert n_features % self.patch_len == 0\n        self.k = n_features // self.patch_len                   # 11\n        cin = self.patch_len                                    # 6 channels\n\n        s = cfg[\"stem_ch\"]\n        self.stems   = nn.ModuleList([cbr(cin, s, 1) for _ in range(4)])\n        self.dense   = DenseNet1d(s, cfg[\"dense_growth\"], cfg[\"dense_layers\"])\n        self.google  = GoogleNet1d(s, cfg[\"incep_modules\"])\n        self.alex    = AlexNet1d(s)\n        self.squeeze = SqueezeNet1d(s, cfg[\"fire_modules\"])\n        comb = self.dense.out_ch + self.google.out_ch + self.alex.out_ch + self.squeeze.out_ch\n\n        self.head = nn.Sequential(                              # Eq. (48)\n            nn.LayerNorm(comb), nn.Dropout(cfg[\"dropout\"]),\n            nn.Linear(comb, 256), nn.ReLU(inplace=True),\n            nn.Dropout(cfg[\"dropout\"]), nn.Linear(256, cfg[\"num_classes\"]))\n\n    def forward(self, x):                                       # (B, n_features)\n        # view THEN transpose: group 6 consecutive columns into a patch, then the\n        # patch index becomes the position axis. view(B, 6, 11) mixes columns wrong.\n        Fm = x.view(x.shape[0], self.k, self.patch_len).transpose(1, 2)   # (B, 6, 11)\n        feats = [gp(stem(Fm)) for stem, gp in\n                 zip(self.stems, [self.dense, self.google, self.alex, self.squeeze])]\n        pooled = [f.mean(dim=-1) for f in feats]                # global average pool\n        return self.head(torch.cat(pooled, dim=1))              # Eq. (47) -> (48)\n\n\nCFG = {\n    \"patch_len\": 6, \"stem_ch\": 96, \"dense_growth\": 32, \"dense_layers\": 3,\n    \"incep_modules\": 2, \"fire_modules\": 3, \"dropout\": 0.1, \"num_classes\": 16,\n}\nN_FEATURES = 66\nN_PARAMS = 395_024\n\n\ndef build_dagsnet(cfg=None, n_features=N_FEATURES):\n    \"\"\"Fresh DAGSNet with PyTorch default initialization; asserts the param budget.\"\"\"\n    model = DAGSNet(cfg or CFG, n_features=n_features)\n    n = sum(p.numel() for p in model.parameters())\n    assert n == N_PARAMS, f\"architecture drift: {n:,} parameters instead of {N_PARAMS:,}\"\n    return model\n",
 "metrics.py": "\"\"\"The 10 metric contract, computed from a summed confusion matrix.\n\nEvaluation is sharded across two GPUs, so the metrics are derived from integer\nconfusion counts rather than from a concatenated prediction vector: integer sums\nare exactly order-independent, float metric averages are not.\n\"\"\"\nimport numpy as np\n\nMETRIC_KEYS = [\n    \"accuracy\",\n    \"precision_macro\", \"precision_micro\", \"precision_weighted\",\n    \"recall_macro\",    \"recall_micro\",    \"recall_weighted\",\n    \"f1_macro\",        \"f1_micro\",        \"f1_weighted\",\n]\n\n\ndef _safe_div(num, den):\n    \"\"\"0/0 -> 0, matching sklearn's zero_division=0.\"\"\"\n    out = np.zeros_like(num, dtype=np.float64)\n    np.divide(num, den, out=out, where=den > 0)\n    return out\n\n\ndef metrics_from_confusion(cm):\n    \"\"\"All 10 metrics from `cm[true, pred]` integer counts.\n\n    Returns the METRIC_KEYS dict; every key is always present even when several\n    of them coincide (they do: micro precision/recall/F1 and weighted recall all\n    equal accuracy in single-label multi-class classification).\n    \"\"\"\n    cm = np.asarray(cm, dtype=np.int64)\n    assert cm.ndim == 2 and cm.shape[0] == cm.shape[1], f\"bad confusion shape {cm.shape}\"\n    n = cm.sum()\n    assert n > 0, \"empty confusion matrix\"\n\n    tp = np.diag(cm).astype(np.float64)\n    support = cm.sum(axis=1).astype(np.float64)          # n_c, true counts\n    predicted = cm.sum(axis=0).astype(np.float64)        # TP_c + FP_c\n\n    precision = _safe_div(tp, predicted)\n    recall = _safe_div(tp, support)\n    f1 = _safe_div(2 * precision * recall, precision + recall)\n\n    accuracy = float(tp.sum() / n)\n    w = support / n\n\n    return {\n        \"accuracy\": accuracy,\n        \"precision_macro\": float(precision.mean()),\n        \"precision_micro\": float(tp.sum() / predicted.sum()),\n        \"precision_weighted\": float((w * precision).sum()),\n        \"recall_macro\": float(recall.mean()),\n        \"recall_micro\": float(tp.sum() / support.sum()),\n        \"recall_weighted\": float((w * recall).sum()),\n        \"f1_macro\": float(f1.mean()),\n        \"f1_micro\": accuracy,          # 2PR/(P+R) with P == R == accuracy\n        \"f1_weighted\": float((w * f1).sum()),\n    }\n\n\ndef per_class_report(cm, class_names):\n    \"\"\"Per-class precision / recall / F1 / support, as a list of dicts.\"\"\"\n    cm = np.asarray(cm, dtype=np.int64)\n    tp = np.diag(cm).astype(np.float64)\n    support = cm.sum(axis=1).astype(np.float64)\n    predicted = cm.sum(axis=0).astype(np.float64)\n    precision = _safe_div(tp, predicted)\n    recall = _safe_div(tp, support)\n    f1 = _safe_div(2 * precision * recall, precision + recall)\n    return [\n        {\"class_index\": i, \"class_name\": class_names[i],\n         \"precision\": float(precision[i]), \"recall\": float(recall[i]),\n         \"f1\": float(f1[i]), \"support\": int(support[i]),\n         \"predicted\": int(predicted[i])}\n        for i in range(len(class_names))\n    ]\n",
 "flatpack.py": "\"\"\"Flat packing of a DAGSNet state_dict.\n\nTwo reasons, both practical:\n\n* A state_dict has 192 entries. Putting one per client on a multiprocessing queue\n  means ~19k shared-memory file descriptors per round at 100 clients, which\n  exhausts the process limit. One tensor per client does not.\n* Aggregation becomes a weighted vector sum instead of a per-key Python loop.\n\nLayout is fixed and parameters come first, so `vec[:n_params]` is exactly the\nlearnable-parameter block the proximal drift is measured over.\n\"\"\"\nimport torch\n\n\nclass Template:\n    \"\"\"Key order, shapes and slices for one architecture.\"\"\"\n\n    def __init__(self, model):\n        sd = model.state_dict()\n        param_names = list(dict(model.named_parameters()))\n        float_keys = param_names + [k for k, v in sd.items()\n                                    if v.is_floating_point() and k not in set(param_names)]\n        int_keys = [k for k, v in sd.items() if not v.is_floating_point()]\n\n        self.float_keys, self.int_keys = float_keys, int_keys\n        self.float_shapes = [tuple(sd[k].shape) for k in float_keys]\n        self.int_shapes = [tuple(sd[k].shape) for k in int_keys]\n        self.float_sizes = [sd[k].numel() for k in float_keys]\n        self.int_sizes = [sd[k].numel() for k in int_keys]\n        self.n_params = sum(sd[k].numel() for k in param_names)\n        self.n_float = sum(self.float_sizes)\n        self.n_int = sum(self.int_sizes)\n        self.int_dtypes = [sd[k].dtype for k in int_keys]\n\n    def flatten(self, sd):\n        f = torch.empty(self.n_float, dtype=torch.float32)\n        i = 0\n        for key, size in zip(self.float_keys, self.float_sizes):\n            f[i:i + size] = sd[key].detach().reshape(-1).to(torch.float32)\n            i += size\n        n = torch.empty(self.n_int, dtype=torch.int64)\n        i = 0\n        for key, size in zip(self.int_keys, self.int_sizes):\n            n[i:i + size] = sd[key].detach().reshape(-1).to(torch.int64)\n            i += size\n        return f, n\n\n    def unflatten(self, f, n):\n        out, i = {}, 0\n        for key, size, shape in zip(self.float_keys, self.float_sizes, self.float_shapes):\n            out[key] = f[i:i + size].reshape(shape).clone()\n            i += size\n        i = 0\n        for key, size, shape, dt in zip(self.int_keys, self.int_sizes, self.int_shapes,\n                                        self.int_dtypes):\n            out[key] = n[i:i + size].reshape(shape).to(dt).clone()\n            i += size\n        return out\n\n\ndef weighted_mean(vecs, weights):\n    \"\"\"Sample-weighted mean in float64: 100 float32 additions lose bits.\"\"\"\n    acc = torch.zeros(vecs[0].numel(), dtype=torch.float64)\n    for v, w in zip(vecs, weights):\n        acc.add_(v.to(torch.float64), alpha=w)\n    return acc.to(torch.float32)\n\n\ndef elementwise_max(vecs):\n    \"\"\"num_batches_tracked is a counter; a weighted mean of it is meaningless.\"\"\"\n    out = vecs[0].clone()\n    for v in vecs[1:]:\n        torch.maximum(out, v, out=out)\n    return out\n",
 "afpha.py": "\"\"\"AFPHA: the approved Proposal A specification (docs/rebuild.md).\n\nEvery constant here completes a gap the paper leaves open. They are implementation\nchoices, not the authors' published hyperparameters, and the two-level averaging is\nalgebraically identical to sample-weighted FedAvg -- say so in any write-up.\n\"\"\"\nimport math\n\nimport numpy as np\nimport torch\n\nimport flatpack\n\nCLUSTER_SIZE = 5\nCLUSTER_SEED = 42\nSHUFFLE_SEED = 42\nMU_BASE = 0.01\nMU_EPS = 1e-12\nLR_MAX = 1e-3\nLR_MIN = 1e-5\nGRAD_CLIP = 1.0\nADAM_BETAS = (0.9, 0.999)\nADAM_EPS = 1e-8\nADAM_WEIGHT_DECAY = 0.0\n\n\ndef build_clusters(num_clients, cluster_size=CLUSTER_SIZE, seed=CLUSTER_SEED):\n    \"\"\"Fixed clusters from one seeded permutation; a short final cluster is kept.\n\n    Clusters are a logical grouping only -- nothing about client ids implies a\n    geographic position or an RSU.\n    \"\"\"\n    perm = np.random.default_rng(seed).permutation(num_clients)\n    return [sorted(int(c) for c in perm[i:i + cluster_size])\n            for i in range(0, num_clients, cluster_size)]\n\n\ndef lr_for_round(rnd, total_rounds):\n    \"\"\"Cosine over communication rounds, fixed within a round. Round index is 1-based.\n\n    A single-round run has no schedule to descend, so it stays at LR_MAX; without this\n    the denominator is zero. The configured run is 50 rounds and never takes that path,\n    but short smoke tests do.\n    \"\"\"\n    assert 1 <= rnd <= total_rounds\n    if total_rounds == 1:\n        return LR_MAX\n    return LR_MIN + (LR_MAX - LR_MIN) / 2 * (1 + math.cos(math.pi * (rnd - 1) / (total_rounds - 1)))\n\n\ndef client_shuffle_seed(rnd, client_id, seed=SHUFFLE_SEED):\n    \"\"\"Deterministic per (round, client) permutation seed.\"\"\"\n    return (seed * 1_000_003 + rnd * 10_007 + client_id) % (2 ** 31 - 1)\n\n\ndef initial_mu(num_clients):\n    return {int(i): MU_BASE for i in range(num_clients)}\n\n\ndef next_mu(drifts, rows):\n    \"\"\"mu_i,t+1 = 0.01 * (1 + d_i / (d_i + d_bar + 1e-12)), d_bar weighted by n_i/N.\n\n    `drifts` and `rows` are dicts keyed by client id. The coefficient stays in\n    [0.01, 0.02): a client that moved further from the new global is held tighter\n    next round. If every drift is zero the rule degenerates to the base value.\n    \"\"\"\n    total = float(sum(rows.values()))\n    d_bar = sum(rows[i] / total * float(drifts[i]) for i in drifts)\n    return {int(i): MU_BASE * (1.0 + float(drifts[i]) / (float(drifts[i]) + d_bar + MU_EPS))\n            for i in drifts}\n\n\n# ------------------------------------------------------------------------ aggregation\n\ndef aggregate(float_vecs, int_vecs, rows):\n    \"\"\"Sample-weighted average of flat state vectors.\n\n    Callers pass a fixed, deterministic order -- never completion order. Float\n    entries (parameters and BatchNorm running stats) are averaged by n_i/N;\n    `num_batches_tracked` counters take the elementwise max.\n    \"\"\"\n    assert len(float_vecs) == len(rows) and float_vecs\n    total = float(sum(rows))\n    weights = [float(r) / total for r in rows]\n    return flatpack.weighted_mean(float_vecs, weights), flatpack.elementwise_max(int_vecs)\n\n\ndef hierarchical_aggregate(client_float, client_int, client_rows, clusters):\n    \"\"\"Cluster average, then server average -- the HFL half of AFPHA.\n\n    Ordering follows `clusters` (sorted ids), so floating-point addition order\n    never depends on which GPU happened to finish first. With both levels\n    sample-weighted this equals sample-weighted FedAvg over all clients; the\n    hierarchy is a system description, not an optimization difference.\n    \"\"\"\n    c_float, c_int, c_rows = [], [], []\n    for members in clusters:\n        f, n = aggregate([client_float[i] for i in members],\n                         [client_int[i] for i in members],\n                         [client_rows[i] for i in members])\n        c_float.append(f); c_int.append(n); c_rows.append(sum(client_rows[i] for i in members))\n    gf, gn = aggregate(c_float, c_int, c_rows)\n    return gf, gn, c_rows\n\n\ndef param_drift(client_float, global_float, n_params):\n    \"\"\"d_i = || w_i - w_global ||_2 over learnable parameters only, in float64.\n\n    Parameters occupy the leading block of the flat layout (see flatpack.Template).\n    \"\"\"\n    d = client_float[:n_params].to(torch.float64) - global_float[:n_params].to(torch.float64)\n    return float(torch.linalg.vector_norm(d))\n",
 "fldata.py": "\"\"\"Data loading for AFPHA-DAGSNet.\n\nContract (see .agents/skills/kaggle-training-notebook/references/veremi-dataset-layout.md):\n\n* Only the 66 `f_*` columns enter the model, in the frozen order from `meta.json`.\n* Train partitions are ALREADY z-scored -- never standardize them a second time.\n* Test is raw and must be scaled with the same train-fitted `scaler.json`.\n* Roots are resolved by a unique sentinel file, not by a fixed mount prefix, so the\n  same code runs against the local tree and the Kaggle mount.\n\"\"\"\nimport hashlib\nimport json\nimport math\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pyarrow.parquet as pq\n\nFP16_MAX = 65504.0\n\n\n# --------------------------------------------------------------------------- paths\n\ndef find_root(sentinel, search_roots, max_depth=5):\n    \"\"\"Locate the unique directory containing `sentinel` under any of `search_roots`.\n\n    `sentinel` is a path relative to the root, e.g. \"20_client/client_stats.json\".\n    Descends up to `max_depth` levels because a Kaggle mount adds an unpredictable\n    prefix. Two shapes have been seen and both must resolve: the flat\n    `/kaggle/input/<dataset>/upload/test/` and the owner-prefixed\n    `/kaggle/input/datasets/<owner>/<dataset>/upload/test/`, which is one level deeper.\n    Raises unless exactly one root matches: an ambiguous mount must fail loudly, not\n    train on the wrong partition.\n    \"\"\"\n    hits, seen = [], set()\n\n    def walk(base, depth):\n        base = Path(base)\n        if not base.is_dir() or base in seen:\n            return\n        seen.add(base)\n        if (base / sentinel).exists():\n            hits.append(base)\n            return                       # do not descend past a match\n        if depth == 0:\n            return\n        for child in sorted(base.iterdir()):\n            if child.is_dir() and not child.name.startswith(\".\"):\n                walk(child, depth - 1)\n\n    for root in search_roots:\n        walk(root, max_depth)\n    hits = sorted({h.resolve() for h in hits})\n    if len(hits) != 1:\n        raise FileNotFoundError(\n            f\"expected exactly one root containing {sentinel!r} under {list(search_roots)}, \"\n            f\"found {len(hits)}: {hits}\")\n    return hits[0]\n\n\n# --------------------------------------------------------------------- frozen schema\n\nclass Schema:\n    \"\"\"Feature order, class names and the train-fitted scaler, loaded once.\"\"\"\n\n    def __init__(self, meta_path, scaler_path, label_mapping_path=None):\n        meta = json.loads(Path(meta_path).read_text())\n        self.features = list(meta[\"feature_cols\"])\n        self.class_names = list(meta[\"class_names\"])\n        self.num_classes = int(meta[\"num_classes\"])\n        assert len(self.features) == 66, f\"expected 66 features, got {len(self.features)}\"\n        assert len(self.class_names) == self.num_classes\n\n        scaler = json.loads(Path(scaler_path).read_text())[\"features\"]\n        self.mean = np.array([scaler[f][\"mean\"] for f in self.features], dtype=np.float64)\n        self.std = np.array([scaler[f][\"std_used\"] for f in self.features], dtype=np.float64)\n        self.fill = np.array([scaler[f].get(\"fill_value\", 0.0) for f in self.features],\n                             dtype=np.float64)\n        assert (self.std > 0).all(), \"scaler has a non-positive std_used\"\n\n        if label_mapping_path is not None:\n            lm = json.loads(Path(label_mapping_path).read_text())\n            assert lm[\"classes\"] == self.class_names, \"label_mapping disagrees with meta.json\"\n\n    def fingerprint(self):\n        payload = json.dumps({\"features\": self.features, \"classes\": self.class_names,\n                              \"mean\": self.mean.tolist(), \"std\": self.std.tolist()},\n                             sort_keys=True)\n        import hashlib\n        return hashlib.sha256(payload.encode()).hexdigest()[:16]\n\n\n# ------------------------------------------------------------------------- reading\n\ndef _read_files(files, schema, out_X, out_y, row0, scale, log=None):\n    \"\"\"Stream `files` into preallocated arrays starting at `row0`; return rows written.\"\"\"\n    cols = schema.features + [\"label\"]\n    n = 0\n    for path in files:\n        pf = pq.ParquetFile(path)\n        for batch in pf.iter_batches(batch_size=262_144, columns=cols, use_threads=True):\n            m = batch.num_rows\n            lo = row0 + n\n            for j, name in enumerate(schema.features):\n                v = batch.column(name).to_numpy(zero_copy_only=False)\n                if scale:\n                    v = np.nan_to_num(v.astype(np.float64), nan=np.nan)\n                    bad = ~np.isfinite(v)\n                    if bad.any():\n                        v[bad] = schema.fill[j]\n                    v = (v - schema.mean[j]) / schema.std[j]\n                out_X[lo:lo + m, j] = v            # numpy downcasts to the array dtype\n            out_y[lo:lo + m] = batch.column(\"label\").to_numpy(zero_copy_only=False)\n            n += m\n        if log:\n            log(f\"    read {Path(path).name}: {n:,} rows so far\")\n    return n\n\n\ndef _check_fp16(X, tag):\n    \"\"\"A z-scored feature stored as fp16 must not overflow; report the observed extreme.\"\"\"\n    peak = float(np.abs(X[:: max(1, len(X) // 2_000_000)]).max())\n    assert np.isfinite(X[:: max(1, len(X) // 2_000_000)]).all(), f\"{tag}: non-finite after cast\"\n    assert peak < FP16_MAX, f\"{tag}: |x|max {peak} exceeds fp16 range\"\n    return peak\n\n\ndef _file_identity(path):\n    \"\"\"Content identity of one parquet file, from its footer only.\n\n    Row counts do not identify data: a file can be rewritten with different features and\n    permuted labels while keeping every count intact, and a fingerprint built on counts\n    alone then declares the new data identical to the old -- reusing a stale prepack cache\n    and permitting a resume onto different mathematics.\n\n    The footer carries enough to tell those apart cheaply: the byte size of the file, the\n    row count, and per row group per column the compressed size plus the min/max/null\n    statistics. Rewriting values moves at least one of those in practice, and reading them\n    costs a footer seek rather than a column decode -- which matters, because this runs\n    before the multi-minute prepack and `--require-resume` has to fail in seconds.\n    \"\"\"\n    md = pq.ParquetFile(path).metadata\n    parts = [path.name, str(path.stat().st_size), str(md.num_rows),\n             str(md.num_row_groups), str(md.num_columns)]\n    for g in range(md.num_row_groups):\n        rg = md.row_group(g)\n        for c in range(rg.num_columns):\n            col = rg.column(c)\n            st = col.statistics\n            parts += [str(col.total_compressed_size),\n                      \"\" if st is None else f\"{st.min}|{st.max}|{st.null_count}\"]\n    return \"\\x1f\".join(parts)\n\n\ndef _digest(items):\n    h = hashlib.sha256()\n    for it in items:\n        h.update(it.encode())\n        h.update(b\"\\x1e\")\n    return h.hexdigest()[:16]\n\n\ndef scan_counts(train_root, test_root):\n    \"\"\"Row counts and content digests from parquet footers only -- no column decode.\n\n    The fingerprint has to cover the data identity, and `--require-resume` has to fail\n    before the multi-minute prepack, so both need these numbers cheaply.\n\n    Returns (rows, n_test, digests) with digests = {\"train\": ..., \"test\": ...}.\n    \"\"\"\n    train_root, test_root = Path(train_root), Path(test_root)\n    client_dirs = sorted(d for d in train_root.iterdir()\n                         if d.is_dir() and d.name.startswith(\"client_id=\"))\n    assert client_dirs, f\"no client_id=* directories under {train_root}\"\n    rows, train_ids = [], []\n    for d in client_dirs:\n        files = sorted(d.glob(\"part-*.parquet\"))\n        assert files, f\"no parquet parts in {d}\"\n        rows.append(sum(pq.ParquetFile(f).metadata.num_rows for f in files))\n        train_ids.append(d.name)\n        train_ids += [_file_identity(f) for f in files]\n    test_files = sorted(test_root.glob(\"part-*.parquet\"))\n    assert test_files, f\"no parquet parts in {test_root}\"\n    n_test = sum(pq.ParquetFile(f).metadata.num_rows for f in test_files)\n    digests = {\"train\": _digest(train_ids),\n               \"test\": _digest(_file_identity(f) for f in test_files)}\n    return np.asarray(rows, dtype=np.int64), int(n_test), digests\n\n\ndef load_clients(train_root, schema, log=print):\n    \"\"\"Read every `client_id=NNN` partition into one contiguous fp16 matrix.\n\n    Returns (X, y, spans, rows) where `spans[i] = (lo, hi)` addresses client i's\n    rows and clients appear in ascending client_id order.\n    \"\"\"\n    train_root = Path(train_root)\n    client_dirs = sorted(d for d in train_root.iterdir()\n                         if d.is_dir() and d.name.startswith(\"client_id=\"))\n    assert client_dirs, f\"no client_id=* directories under {train_root}\"\n\n    per_client_files, rows = [], []\n    for d in client_dirs:\n        files = sorted(d.glob(\"part-*.parquet\"))\n        assert files, f\"no parquet parts in {d}\"\n        per_client_files.append(files)\n        rows.append(sum(pq.ParquetFile(f).metadata.num_rows for f in files))\n    total = int(sum(rows))\n    log(f\"  {len(client_dirs)} clients, {total:,} rows, \"\n        f\"{total * 66 * 2 / 2**30:.2f} GiB as fp16\")\n\n    X = np.empty((total, 66), dtype=np.float16)\n    y = np.empty(total, dtype=np.uint8)\n    spans, cursor, t0 = [], 0, time.perf_counter()\n    for i, files in enumerate(per_client_files):\n        got = _read_files(files, schema, X, y, cursor, scale=False)\n        assert got == rows[i], f\"client {i}: expected {rows[i]} rows, read {got}\"\n        spans.append((cursor, cursor + got))\n        cursor += got\n        if (i + 1) % 10 == 0 or i == len(per_client_files) - 1:\n            log(f\"  client {i + 1}/{len(per_client_files)}  \"\n                f\"{cursor:,} rows  {time.perf_counter() - t0:.0f}s\")\n    assert cursor == total\n    peak = _check_fp16(X, \"train\")\n    assert y.max() < schema.num_classes, \"train label out of range\"\n    log(f\"  train ready in {time.perf_counter() - t0:.0f}s, |x|max~{peak:.1f}, \"\n        f\"classes present {len(np.unique(y))}/{schema.num_classes}\")\n    return X, y, spans, np.asarray(rows, dtype=np.int64)\n\n\ndef load_test(test_root, schema, log=print):\n    \"\"\"Read the fixed test set and apply the train-fitted scaler.\"\"\"\n    test_root = Path(test_root)\n    files = sorted(test_root.glob(\"part-*.parquet\"))\n    assert files, f\"no parquet parts in {test_root}\"\n    total = int(sum(pq.ParquetFile(f).metadata.num_rows for f in files))\n    log(f\"  test {total:,} rows from {len(files)} files\")\n    X = np.empty((total, 66), dtype=np.float16)\n    y = np.empty(total, dtype=np.uint8)\n    t0 = time.perf_counter()\n    got = _read_files(files, schema, X, y, 0, scale=True)\n    assert got == total\n    peak = _check_fp16(X, \"test\")\n    assert set(np.unique(y).tolist()) == set(range(schema.num_classes)), \\\n        \"test does not contain all 16 classes\"\n    log(f\"  test ready in {time.perf_counter() - t0:.0f}s, |x|max~{peak:.1f}\")\n    return X, y\n\n\ndef steps_per_round(rows, batch):\n    \"\"\"Optimizer steps a full round costs; the tail batch is a real step.\"\"\"\n    return int(sum(math.ceil(int(r) / batch) for r in rows))\n",
 "worker.py": "\"\"\"One persistent process per GPU.\n\nEach worker holds its own resident copy of the whole training matrix and of its\ndisjoint test shard, and a compiled DAGSNet. Clients are handed out dynamically\nfrom a shared queue, so a slow client never strands a GPU; the parent -- not the\ncompletion order -- fixes the aggregation order.\n\nDifferent clients are different models. Nothing here forms a process group and no\ngradient is ever synchronized between GPUs.\n\"\"\"\nimport math\nimport time\nimport traceback\n\nimport numpy as np\nimport torch\n\nimport flatpack\nfrom afpha import ADAM_BETAS, ADAM_EPS, ADAM_WEIGHT_DECAY, GRAD_CLIP, client_shuffle_seed\nfrom dagsnet import build_dagsnet\n\nCOMPILE_FALLBACK = [\"reduce-overhead\", \"default\", \"eager\"]\n\n\ndef _to_device_chunked(arr, device, chunk=4_000_000):\n    \"\"\"Copy a mmapped array to the GPU without materializing a host copy.\"\"\"\n    out = torch.empty(arr.shape, dtype=torch.float16 if arr.dtype == np.float16\n                      else torch.uint8, device=device)\n    for i in range(0, len(arr), chunk):\n        out[i:i + chunk] = torch.from_numpy(np.ascontiguousarray(arr[i:i + chunk])).to(\n            device, non_blocking=True)\n    return out\n\n\nclass Trainer:\n    \"\"\"Model, compiled graph and resident tensors for one GPU.\"\"\"\n\n    def __init__(self, device, batch, compile_mode, log):\n        self.device = device\n        self.batch = batch\n        self.log = log\n        self.model = build_dagsnet().to(device)\n        self.params = list(self.model.parameters())\n        self.eager = self.model\n        self.compiled = self.model\n        self.compile_mode = \"eager\"\n        self.crit = torch.nn.CrossEntropyLoss()\n        self.template = flatpack.Template(self.eager)\n        # Skipped AMP steps are counted by watching the scale tensor without a host sync\n        # (get_scale() would .item() every step). _scale is created lazily on the first\n        # scale() call, so probe it that way; fail loudly if a torch upgrade removes it.\n        probe = torch.amp.GradScaler(\"cuda\")\n        probe.scale(torch.zeros(1, device=device))\n        assert isinstance(getattr(probe, \"_scale\", None), torch.Tensor), \\\n            \"GradScaler._scale is unavailable; skipped-step counting needs a new mechanism\"\n        del probe\n        if compile_mode != \"eager\":\n            self._try_compile(compile_mode)\n\n    def _try_compile(self, requested):\n        \"\"\"Compile, then prove it with a real step and a logit comparison.\n\n        torch.compile is lazy, so a bare try around the call catches nothing --\n        the trial forward/backward below is what actually validates the backend.\n        The logit check runs in eval() so Dropout cannot make two forwards differ,\n        and the trial steps' BatchNorm drift is undone from a snapshot afterwards.\n        \"\"\"\n        order = COMPILE_FALLBACK[COMPILE_FALLBACK.index(requested):]\n        snapshot = {k: v.detach().clone() for k, v in self.model.state_dict().items()}\n        x = torch.randn(self.batch, 66, device=self.device, dtype=torch.float16)\n        y = torch.randint(0, 16, (self.batch,), device=self.device)\n\n        def restore():\n            self.model.load_state_dict(snapshot, strict=True)\n            self.model.zero_grad(set_to_none=True)\n            self.model.train()\n\n        self.model.eval()\n        with torch.no_grad(), torch.autocast(\"cuda\", dtype=torch.float16):\n            ref = self.eager(x).float().clone()\n        self.model.train()\n\n        for mode in order:\n            if mode == \"eager\":\n                break\n            try:\n                cand = torch.compile(self.model, mode=None if mode == \"default\" else mode)\n                # Mirror the real step exactly, zero_grad included. Consecutive backwards\n                # without it accumulate into CUDA-graph-owned .grad buffers and raise\n                # \"gradient tensor output of CUDAGraphs ... overwritten by a subsequent run\".\n                opt = torch.optim.Adam(self.params, lr=1e-8, fused=True)\n                sc = torch.amp.GradScaler(\"cuda\")\n                for _ in range(4):\n                    with torch.autocast(\"cuda\", dtype=torch.float16):\n                        sc.scale(self.crit(cand(x), y)).backward()\n                    sc.unscale_(opt)\n                    torch.nn.utils.clip_grad_norm_(self.params, 1.0)\n                    sc.step(opt); sc.update()\n                    opt.zero_grad(set_to_none=True)\n                del opt, sc\n                restore()\n\n                self.model.eval()\n                with torch.no_grad(), torch.autocast(\"cuda\", dtype=torch.float16):\n                    got = cand(x).float().clone()\n                self.model.train()\n                delta = (got - ref).abs().max().item()\n                # Argmax on a freshly initialized model is decided by near-ties well below\n                # fp16 resolution, so compare only rows whose top-2 gap is clearly larger\n                # than the observed numerical difference.\n                top2 = ref.topk(2, dim=1).values\n                decisive = (top2[:, 0] - top2[:, 1]) > max(10 * delta, 1e-3)\n                # Count disagreements as an integer. torch.mean on CUDA multiplies by a\n                # rounded reciprocal, so a perfect match returns 0.99999994 for many row\n                # counts (77 of the first 600) and a `< 1.0` test rejects a correct build\n                # at random -- silently costing the CUDA-graph speedup.\n                n_dec = int(decisive.sum())\n                n_bad = int((got.argmax(1) != ref.argmax(1))[decisive].sum())\n                if not math.isfinite(delta) or delta > 1e-2 or n_bad:\n                    raise RuntimeError(\n                        f\"compiled logits diverge: max|d|={delta}, \"\n                        f\"{n_bad} argmax disagreements over {n_dec}/{len(ref)} decisive rows\")\n                restore()\n                self.compiled, self.compile_mode = cand, mode\n                self.log(f\"    compile mode={mode} ok (max|dlogit|={delta:.2e}, \"\n                         f\"0 argmax disagreements on {n_dec}/{len(ref)} decisive rows)\")\n                return\n            except Exception as exc:                       # noqa: BLE001 - fall back on anything\n                self.log(f\"    compile mode={mode} rejected: {type(exc).__name__}: {exc}\")\n                restore()\n        self.compiled, self.compile_mode = self.eager, \"eager\"\n        self.log(\"    running eager\")\n\n    def train_client(self, X, Y, lo, hi, rnd, cid, lr, mu, w_global):\n        \"\"\"One full local epoch over rows [lo, hi). Returns (state_dict, stats).\"\"\"\n        t0 = time.perf_counter()\n        model = self.model\n        model.train()\n        opt = torch.optim.Adam(self.params, lr=lr, betas=ADAM_BETAS, eps=ADAM_EPS,\n                               weight_decay=ADAM_WEIGHT_DECAY, fused=True)\n        scaler = torch.amp.GradScaler(\"cuda\")\n\n        seed = client_shuffle_seed(rnd, cid)\n        g = torch.Generator(device=self.device)\n        g.manual_seed(seed)\n        perm = lo + torch.randperm(hi - lo, generator=g, device=self.device)\n        # Dropout draws from the global generators, not from `g`. Without seeding them\n        # per (round, client) the mask sequence depends on which GPU picked the client\n        # up and on how many clients ran before it, so a rescheduled run would differ\n        # for a reason that has nothing to do with the algorithm.\n        torch.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n\n        n = hi - lo\n        full = (n // self.batch) * self.batch          # graph-captured steps\n        steps = 0\n        # Accumulate on the device and read once at the end. Two float() calls per step\n        # cost a measured 13% (8.12 vs 7.20 ms/step) by forcing extra host syncs.\n        # Accumulate only over APPLIED steps. A step the GradScaler skipped overflowed,\n        # so its gradient norm is inf by construction and its loss may be nan; folding\n        # those into the mean turns a normal AMP warm-up into a fake divergence signal.\n        ce_acc = torch.zeros((), device=self.device)\n        gn_acc = torch.zeros((), device=self.device)\n        skip_acc = torch.zeros((), device=self.device)\n        zero = torch.zeros((), device=self.device)\n\n        for i in range(0, n, self.batch):\n            idx = perm[i:i + self.batch]\n            # The tail batch is a real optimizer step and must not be dropped, but its\n            # shape would force a recompile -- run that one step on the eager module.\n            net = self.compiled if i < full else self.eager\n            with torch.autocast(\"cuda\", dtype=torch.float16):\n                loss = self.crit(net(X[idx]), Y[idx])\n            scaler.scale(loss).backward()\n            scaler.unscale_(opt)\n            # d/dw of (mu/2)||w - w_global||^2 == mu (w - w_global); added after unscale so\n            # it is in true gradient units, and before clipping so the clip sees both terms.\n            # rebuilt every step: zero_grad(set_to_none=True) replaces the .grad objects\n            torch._foreach_add_([p.grad for p in self.params],\n                                torch._foreach_sub(self.params, w_global), alpha=mu)\n            gn = torch.nn.utils.clip_grad_norm_(self.params, GRAD_CLIP)\n            prev = scaler._scale.clone()\n            scaler.step(opt)\n            scaler.update()\n            applied = scaler._scale >= prev         # a halved scale means the step was skipped\n            skip_acc += ~applied\n            opt.zero_grad(set_to_none=True)\n            # torch.where, not multiplication: inf * 0 is nan.\n            ce_acc += torch.where(applied, loss.detach(), zero)\n            gn_acc += torch.where(applied, gn, zero)\n            steps += 1\n\n        ce_sum, gn_sum, skipped = float(ce_acc), float(gn_acc), int(skip_acc)\n        applied_steps = steps - skipped\n        ce_mean = ce_sum / max(applied_steps, 1)\n        gn_mean = gn_sum / max(applied_steps, 1)\n\n        vec_f, vec_i = self.template.flatten(model.state_dict())\n        finite = bool(torch.isfinite(vec_f).all())\n        stats = {\"client_id\": cid, \"rows\": int(n), \"steps\": steps, \"skipped\": skipped,\n                 \"applied_steps\": applied_steps,\n                 \"skip_pct\": 100.0 * skipped / max(steps, 1),\n                 # means over applied steps only; see the accumulator comment above\n                 \"ce_mean\": ce_mean, \"grad_norm_mean\": gn_mean,\n                 \"lr\": lr, \"mu\": mu, \"seed\": seed,\n                 # Three separate health facts. Finite weights alone do not mean the\n                 # client trained: every step can be skipped and still return the\n                 # global weights unchanged and finite.\n                 \"finite_weights\": finite,\n                 \"finite_loss\": math.isfinite(ce_mean),\n                 \"finite_grad\": math.isfinite(gn_mean),\n                 \"seconds\": time.perf_counter() - t0, \"device\": str(self.device)}\n        return vec_f, vec_i, stats\n\n    @torch.inference_mode()\n    def evaluate(self, X, Y, num_classes, eval_batch):\n        \"\"\"Confusion counts and predictions for this worker's contiguous test shard.\"\"\"\n        t0 = time.perf_counter()\n        self.model.eval()\n        conf = torch.zeros(num_classes * num_classes, dtype=torch.int64, device=self.device)\n        preds = torch.empty(len(X), dtype=torch.uint8, device=self.device)\n        full = (len(X) // eval_batch) * eval_batch\n        for i in range(0, len(X), eval_batch):\n            net = self.compiled if i < full else self.eager\n            with torch.autocast(\"cuda\", dtype=torch.float16):\n                logits = net(X[i:i + eval_batch])\n            p = logits.float().argmax(1)               # graph output: consumed immediately\n            preds[i:i + eval_batch] = p.to(torch.uint8)\n            conf += torch.bincount(Y[i:i + eval_batch].long() * num_classes + p,\n                                   minlength=num_classes * num_classes)\n        self.model.train()\n        return (conf.view(num_classes, num_classes).cpu().numpy(),\n                preds.cpu().numpy(), time.perf_counter() - t0)\n\n\ndef worker_main(rank, dev_index, cfg, paths, shard, ctrl_q, task_q, res_q):\n    \"\"\"Process entry point. Never raises past the reporting boundary.\"\"\"\n    def log(msg):\n        print(f\"[gpu{rank}] {msg}\", flush=True)\n\n    try:\n        torch.set_num_threads(1)\n        torch.backends.cudnn.benchmark = True\n        device = torch.device(f\"cuda:{dev_index}\")\n        torch.cuda.set_device(device)\n        log(f\"{torch.cuda.get_device_name(device)} cap={torch.cuda.get_device_capability(device)}\")\n\n        Xm = np.load(paths[\"train_X\"], mmap_mode=\"r\")\n        Ym = np.load(paths[\"train_y\"], mmap_mode=\"r\")\n        Xt = np.load(paths[\"test_X\"], mmap_mode=\"r\")\n        Yt = np.load(paths[\"test_y\"], mmap_mode=\"r\")\n        t0 = time.perf_counter()\n        X = _to_device_chunked(Xm, device)\n        Y = _to_device_chunked(Ym, device).long()\n        lo, hi = shard\n        Xs = _to_device_chunked(Xt[lo:hi], device)\n        Ys = _to_device_chunked(Yt[lo:hi], device).long()\n        log(f\"resident: train {X.shape} + test shard {Xs.shape} in {time.perf_counter()-t0:.0f}s, \"\n            f\"{torch.cuda.memory_allocated(device)/2**30:.2f} GiB allocated\")\n\n        tr = Trainer(device, cfg[\"batch\"], cfg[\"compile_mode\"], log)\n        free, total = torch.cuda.mem_get_info(device)\n        res_q.put((\"ready\", rank, None,\n                   {\"compile_mode\": tr.compile_mode,\n                    \"device_name\": torch.cuda.get_device_name(device),\n                    \"capability\": list(torch.cuda.get_device_capability(device)),\n                    \"free_GiB\": free / 2**30, \"total_GiB\": total / 2**30,\n                    \"resident_GiB\": torch.cuda.memory_allocated(device) / 2**30}))\n\n        while True:\n            msg = ctrl_q.get()\n            kind = msg[0]\n            if kind == \"stop\":\n                break\n            if kind == \"round\":\n                _, rnd, lr, mus, gf, gi = msg\n                global_sd = {k: v.to(device) for k, v in tr.template.unflatten(gf, gi).items()}\n                tr.model.load_state_dict(global_sd, strict=True)\n                w_global = [p.detach().clone() for p in tr.params]\n                for cid in iter(task_q.get, None):\n                    tr.model.load_state_dict(global_sd, strict=True)\n                    vec_f, vec_i, stats = tr.train_client(\n                        X, Y, *cfg[\"spans\"][cid], rnd, cid, lr, mus[cid], w_global)\n                    res_q.put((\"client\", rank, (vec_f, vec_i), stats))\n                res_q.put((\"round_done\", rank, None, None))\n            elif kind == \"eval\":\n                _, rnd, gf, gi = msg\n                tr.model.load_state_dict(\n                    {k: v.to(device) for k, v in tr.template.unflatten(gf, gi).items()},\n                    strict=True)\n                conf, preds, secs = tr.evaluate(Xs, Ys, cfg[\"num_classes\"], cfg[\"eval_batch\"])\n                res_q.put((\"eval\", rank, conf, {\"round\": rnd, \"seconds\": secs,\n                                                \"shard\": [lo, hi]}))\n                res_q.put((\"preds\", rank, preds, None))\n            elif kind == \"peak\":\n                res_q.put((\"peak\", rank, None,\n                           {\"peak_GiB\": torch.cuda.max_memory_allocated(device) / 2**30}))\n    except Exception:                                   # noqa: BLE001 - report, never hang\n        res_q.put((\"error\", rank, None, {\"traceback\": traceback.format_exc()}))\n        raise\n",
 "fl_train.py": "\"\"\"AFPHA-DAGSNet federated training driver.\n\nOne round: broadcast the global weights, train every client for one full local\nepoch on its own GPU-resident rows, aggregate cluster-then-server by sample count,\nevaluate the new global model on the entire fixed test set, commit the artifacts.\n\nNothing here uses the test set to steer training.\n\"\"\"\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport queue\nimport shutil\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.multiprocessing as mp\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\n\nimport afpha\nimport fldata\nimport flatpack\nfrom dagsnet import CFG as MODEL_CFG, N_FEATURES, N_PARAMS, build_dagsnet\nfrom metrics import METRIC_KEYS, metrics_from_confusion, per_class_report\nfrom worker import worker_main\n\nTOTAL_ROUNDS = 50\n\n\n# ------------------------------------------------------------------------- config\n\ndef scenario_config(num_clients):\n    return {\"num_clients\": num_clients,\n            \"batch\": 512 if num_clients in (20, 50) else 256,\n            \"run_name\": f\"afpha-dagsnet-{num_clients}client\"}\n\n\ndef resolve_paths(num_clients, args):\n    \"\"\"Resolve the partition root and the test root by sentinel, not by mount prefix.\"\"\"\n    train_root = fldata.find_root(f\"{num_clients}_client/client_stats.json\",\n                                  args.data_search) / f\"{num_clients}_client\"\n    test_base = fldata.find_root(\"test/part-00000.parquet\", args.test_search)\n    return train_root, test_base\n\n\n# --------------------------------------------------------------------- artifacts\n\nclass Run:\n    \"\"\"Output tree. A round is only 'complete' once its marker exists.\"\"\"\n\n    def __init__(self, root):\n        self.root = Path(root)\n        for sub in (\"weights\", \"metrics\", \"confusion\", \"per_class\", \"preds\",\n                    \"client_log\", \"complete\", \"model_source\", \"resume\"):\n            (self.root / sub).mkdir(parents=True, exist_ok=True)\n        self.history = self.root / \"metrics\" / \"history.csv\"\n        self.heartbeat = self.root / \"heartbeat.jsonl\"\n\n    def last_complete_round(self):\n        done = sorted(int(p.stem.split(\"_\")[1])\n                      for p in (self.root / \"complete\").glob(\"round_*.done\"))\n        return done[-1] if done else 0\n\n    def emit(self, record, wb=None, payload=None, step=None):\n        with self.heartbeat.open(\"a\") as fh:\n            fh.write(json.dumps(record) + \"\\n\")\n        print(\"  \" + json.dumps(record), flush=True)\n        if wb is not None and payload is not None:\n            wb.log(payload, step=step)\n\n    def history_row_from_json(self, rnd):\n        \"\"\"Rebuild one history row from the round's committed metrics JSON.\n\n        The CSV is derived data: every column comes from `metrics/round_NNN.json`, which\n        is written inside the same commit. That makes the CSV reconstructible, which is\n        what lets `append_history` heal a truncated file instead of propagating the loss.\n        \"\"\"\n        d = json.loads((self.root / \"metrics\" / f\"round_{rnd:03d}.json\").read_text())\n        return {\"round\": str(rnd),\n                **{k: f\"{float(d[k]):.6f}\" for k in METRIC_KEYS},\n                **{f\"t_{k}\": f\"{float(v):.1f}\" for k, v in d[\"timing\"].items()}}\n\n    def append_history(self, row):\n        \"\"\"Idempotent: a re-run round replaces its row instead of duplicating it.\n\n        Two failure modes are handled here, both observed:\n\n        * The whole file is rewritten on every round, so a crash partway through used to\n          leave a CSV holding only the rounds written before the interruption -- committed\n          history was lost even though the markers and the per-round JSON survived. The\n          write now goes to a temporary file and is published with `os.replace`, which is\n          atomic on POSIX: a reader sees either the old file or the new one.\n        * A CSV truncated by an *earlier* crash would otherwise stay truncated forever.\n          Before writing, every round whose marker exists is restored from its JSON, so\n          the next commit repairs the file rather than carrying the gap forward.\n        \"\"\"\n        rows = {}\n        if self.history.exists():\n            with self.history.open() as fh:\n                for r in csv.DictReader(fh):\n                    rows[int(r[\"round\"])] = r\n        for rnd in sorted(int(p.stem.split(\"_\")[1])\n                          for p in (self.root / \"complete\").glob(\"round_*.done\")):\n            if rnd not in rows:\n                try:\n                    rows[rnd] = self.history_row_from_json(rnd)\n                except (OSError, KeyError, ValueError, json.JSONDecodeError):\n                    pass          # unreadable JSON is a separate problem; do not mask it\n        rows[int(row[\"round\"])] = {k: str(v) for k, v in row.items()}\n        fields = list(row)\n        tmp = self.history.with_name(self.history.name + \".tmp\")\n        with tmp.open(\"w\", newline=\"\") as fh:\n            w = csv.DictWriter(fh, fieldnames=fields)\n            w.writeheader()\n            for k in sorted(rows):\n                w.writerow({f: rows[k].get(f, \"\") for f in fields})\n            fh.flush()\n            os.fsync(fh.fileno())\n        os.replace(tmp, self.history)\n\n    def commit(self, rnd, global_sd, cm, preds, client_stats, cluster_rows, mu_next, drifts,\n               timing, schema, resume_state):\n        \"\"\"Write everything, then the marker. A crash mid-write leaves the round incomplete.\n\n        The resume state is per-round and written before the marker, so the state a\n        resume reads always belongs to the round the marker certifies. A single latest\n        resume_state.pt cannot give that guarantee: a crash between the marker write\n        and the state write leaves the two describing different rounds.\n        \"\"\"\n        m = metrics_from_confusion(cm)\n        torch.save({k: v for k, v in global_sd.items()},\n                   self.root / \"weights\" / f\"round_{rnd:03d}.pt\")\n        np.save(self.root / \"confusion\" / f\"round_{rnd:03d}.npy\", cm)\n        np.save(self.root / \"preds\" / f\"round_{rnd:03d}_ypred.npy\", preds)\n        (self.root / \"per_class\" / f\"round_{rnd:03d}.json\").write_text(\n            json.dumps(per_class_report(cm, schema.class_names), indent=2))\n        (self.root / \"metrics\" / f\"round_{rnd:03d}.json\").write_text(\n            json.dumps({\"round\": rnd, **m, \"timing\": timing,\n                        \"test_rows\": int(cm.sum())}, indent=2))\n        (self.root / \"client_log\" / f\"round_{rnd:03d}.json\").write_text(\n            json.dumps({\"round\": rnd, \"clients\": client_stats, \"cluster_rows\": cluster_rows,\n                        \"mu_next\": mu_next, \"drift\": drifts}, indent=2))\n        self.append_history({\"round\": rnd, **{k: f\"{m[k]:.6f}\" for k in METRIC_KEYS},\n                             **{f\"t_{k}\": f\"{v:.1f}\" for k, v in timing.items()}})\n        torch.save(resume_state, self.root / \"resume\" / f\"round_{rnd:03d}.pt\")\n        (self.root / \"complete\" / f\"round_{rnd:03d}.done\").write_text(\n            json.dumps({\"round\": rnd, \"utc\": time.strftime(\"%Y-%m-%dT%H:%M:%SZ\", time.gmtime())}))\n        return m\n\n\ndef import_previous(run, search_roots, fingerprint, log, explicit=None):\n    \"\"\"Copy a prior session's run directory into this session's empty working dir.\n\n    A new Kaggle session starts with an empty /kaggle/working, so the previous run\n    reaches it read-only through `kernel_sources` under /kaggle/input. Without this\n    step `last_complete_round()` sees nothing and round 1 starts over. Only a run whose\n    config.json carries the same fingerprint is eligible: a different scientific setting\n    must not be silently continued.\n    \"\"\"\n    if run.last_complete_round():\n        return 0\n    found = []\n    for root in search_roots:\n        root = Path(root)\n        if not root.is_dir():\n            continue\n        for cfg_path in sorted(root.glob(\"**/config.json\")):\n            src = cfg_path.parent\n            if src.resolve() == run.root.resolve():\n                continue\n            try:\n                if json.loads(cfg_path.read_text()).get(\"fingerprint\") != fingerprint:\n                    continue\n                done = sorted(int(q.stem.split(\"_\")[1])\n                              for q in (src / \"complete\").glob(\"round_*.done\"))\n            except (OSError, ValueError, json.JSONDecodeError):\n                continue\n            if done:\n                found.append((src, done[-1]))\n    if explicit is not None:\n        want = Path(explicit).resolve()\n        found = [(s_, n) for s_, n in found if s_.resolve() == want]\n        assert found, f\"--resume-source {explicit} holds no completed round with fingerprint {fingerprint}\"\n    if not found:\n        return 0\n    # The fingerprint identifies the *configuration*, not the run. Two independent\n    # trainings of the same scenario share it while holding different weights, so\n    # picking the one with the most rounds would silently splice two histories\n    # together. Ambiguity is the user's to resolve.\n    if len(found) > 1:\n        listing = \"\\n  \".join(f\"{s_} ({n} rounds)\" for s_, n in sorted(found))\n        raise SystemExit(\n            f\"{len(found)} candidate resume sources share fingerprint {fingerprint}:\\n\"\n            f\"  {listing}\\n\"\n            \"Refusing to guess which run to continue. Pass --resume-source <dir> to choose.\")\n    src, last = found[0]\n    log(f\"importing {last} completed rounds from {src}\")\n    # Markers are copied last and only after the bundle validates. A marker is the claim\n    # 'this round is complete and its artifacts are here'; publishing it while the copy is\n    # still in flight makes an interrupted import look like a finished one, and the retry\n    # then returns immediately with weights still missing.\n    markers = []\n    for item in sorted(src.rglob(\"*\")):\n        if not item.is_file():\n            continue\n        rel = item.relative_to(src)\n        if rel.parts and rel.parts[0] == \"complete\":\n            markers.append((item, rel))\n            continue\n        dst = run.root / rel\n        dst.parent.mkdir(parents=True, exist_ok=True)\n        # A same-size file is treated as already imported; a different size means the\n        # previous attempt was cut off mid-file, so redo it rather than skipping it.\n        if dst.exists() and dst.stat().st_size == item.stat().st_size:\n            continue\n        # copyfile, not copy2: the source is a read-only mount (/kaggle/input), and\n        # copy2 would carry r--r--r-- across. config.json and history.csv are\n        # rewritten every round, so the next write would fail with PermissionError.\n        part = dst.with_name(dst.name + \".part\")\n        shutil.copyfile(item, part)\n        part.chmod(0o644)\n        os.replace(part, dst)\n    missing = bundle_gaps(run.root, last)\n    assert not missing, (\n        f\"import from {src} is incomplete; refusing to publish completion markers. \"\n        f\"missing: {missing[:8]}{' ...' if len(missing) > 8 else ''}\")\n    for item, rel in sorted(markers):\n        dst = run.root / rel\n        dst.parent.mkdir(parents=True, exist_ok=True)\n        part = dst.with_name(dst.name + \".part\")\n        shutil.copyfile(item, part)\n        part.chmod(0o644)\n        os.replace(part, dst)\n    got = run.last_complete_round()\n    assert got == last, f\"imported {last} rounds but only {got} are complete after copy\"\n    return got\n\n\nROUND_ARTIFACTS = ((\"weights\", \"pt\"), (\"metrics\", \"json\"), (\"confusion\", \"npy\"),\n                   (\"per_class\", \"json\"), (\"client_log\", \"json\"), (\"resume\", \"pt\"))\n\n\ndef bundle_gaps(root, last):\n    \"\"\"Every file a round is supposed to have, for rounds 1..last. Empty list == complete.\"\"\"\n    root = Path(root)\n    missing = []\n    for name in (\"config.json\", \"preds/y_true.npy\"):\n        if not (root / name).exists():\n            missing.append(name)\n    for rnd in range(1, last + 1):\n        for sub, ext in ROUND_ARTIFACTS:\n            p = root / sub / f\"round_{rnd:03d}.{ext}\"\n            if not p.exists() or p.stat().st_size == 0:\n                missing.append(f\"{sub}/round_{rnd:03d}.{ext}\")\n        p = root / \"preds\" / f\"round_{rnd:03d}_ypred.npy\"\n        if not p.exists() or p.stat().st_size == 0:\n            missing.append(f\"preds/round_{rnd:03d}_ypred.npy\")\n    return missing\n\n\n# ------------------------------------------------------------------------ prepack\n\ndef prepack(train_root, test_base, schema, scratch, rows, n_test, digests, log):\n    \"\"\"Parquet -> contiguous .npy once per session; workers mmap these.\n\n    The cache key covers the schema, both roots and the row counts. Keying on the train\n    root alone would reuse a stale test array after the test source changed.\n    \"\"\"\n    scratch = Path(scratch)\n    scratch.mkdir(parents=True, exist_ok=True)\n    paths = {k: str(scratch / f\"{k}.npy\")\n             for k in (\"train_X\", \"train_y\", \"test_X\", \"test_y\")}\n    # The digests are what make a rewritten source at the same path a cache miss; row\n    # counts and paths alone cannot see a file whose values changed but whose shape did not.\n    key = {\"schema\": schema.fingerprint(), \"train_root\": str(train_root),\n           \"test_root\": str(test_base), \"rows\": rows.tolist(), \"n_test\": int(n_test),\n           \"digests\": digests}\n    meta_path = scratch / \"prepack.json\"\n    if meta_path.exists() and all(Path(p).exists() for p in paths.values()):\n        info = json.loads(meta_path.read_text())\n        if info.get(\"key\") == key:\n            log(\"  prepack cache hit\")\n            return paths, info[\"spans\"]\n\n    t0 = time.perf_counter()\n    log(\"  reading train partitions\")\n    X, y, spans, got_rows = fldata.load_clients(train_root / \"train\", schema, log=log)\n    assert got_rows.tolist() == rows.tolist(), \"row counts changed between scan and decode\"\n    np.save(paths[\"train_X\"], X); np.save(paths[\"train_y\"], y)\n    del X, y\n    t_train = time.perf_counter() - t0\n    log(\"  reading test\")\n    t1 = time.perf_counter()\n    Xt, yt = fldata.load_test(test_base / \"test\", schema, log=log)\n    assert len(yt) == n_test, f\"test rows changed between scan ({n_test}) and decode ({len(yt)})\"\n    np.save(paths[\"test_X\"], Xt); np.save(paths[\"test_y\"], yt)\n    del Xt, yt\n    meta_path.write_text(json.dumps({\"key\": key, \"spans\": spans,\n                                     \"train_decode_s\": t_train,\n                                     \"test_decode_s\": time.perf_counter() - t1,\n                                     \"seconds\": time.perf_counter() - t0}))\n    log(f\"  prepack done in {time.perf_counter() - t0:.0f}s \"\n        f\"(train {t_train:.0f}s, test {time.perf_counter() - t1:.0f}s)\")\n    return paths, spans\n\n\n# --------------------------------------------------------------------------- main\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--clients\", type=int, required=True, choices=(20, 50, 100))\n    ap.add_argument(\"--out\", required=True)\n    ap.add_argument(\"--scratch\", required=True)\n    ap.add_argument(\"--arch\", required=True, help=\"dir holding meta.json and scaler.json\")\n    ap.add_argument(\"--data-search\", nargs=\"+\", required=True)\n    ap.add_argument(\"--test-search\", nargs=\"+\", required=True)\n    ap.add_argument(\"--rounds\", type=int, default=TOTAL_ROUNDS)\n    ap.add_argument(\"--eval-batch\", type=int, default=16384)\n    ap.add_argument(\"--compile-mode\", default=\"reduce-overhead\",\n                    choices=(\"reduce-overhead\", \"default\", \"eager\"))\n    ap.add_argument(\"--max-hours\", type=float, default=11.0)\n    ap.add_argument(\"--heartbeat-clients\", type=int, default=5)\n    ap.add_argument(\"--resume-search\", nargs=\"*\", default=[],\n                    help=\"Dirs holding a previous session's run output (Kaggle: /kaggle/input). \"\n                         \"Only a run with a matching fingerprint is imported.\")\n    ap.add_argument(\"--resume-source\", default=None,\n                    help=\"Import from exactly this run directory. Required when several \"\n                         \"candidates under --resume-search share the fingerprint.\")\n    ap.add_argument(\"--require-resume\", action=\"store_true\",\n                    help=\"Fail before prepack if no matching previous run is found.\")\n    ap.add_argument(\"--max-skip-pct\", type=float, default=5.0,\n                    help=\"Abort the round if a client skipped more than this share of its AMP \"\n                         \"steps, beyond the --max-skip-abs warm-up allowance.\")\n    ap.add_argument(\"--max-skip-abs\", type=int, default=8,\n                    help=\"Skips always tolerated per client per round. The GradScaler is reset \"\n                         \"for every client, so each one walks its scale down from 65536 and \"\n                         \"skips a few steps doing so; that is calibration, not divergence. \"\n                         \"Production clients run 384-11,506 steps, so this is well under 1%.\")\n    ap.add_argument(\"--worker-devices\", type=int, nargs=\"+\", default=None,\n                    help=\"CUDA indices, one worker each. Default: one per GPU, capped at 2. \"\n                         \"Repeat an index to exercise the two-worker protocol on one GPU.\")\n    ap.add_argument(\"--wandb-project\", default=\"afpha-dagsnet-veremi\")\n    ap.add_argument(\"--wandb-entity\", default=None)\n    ap.add_argument(\"--no-wandb\", action=\"store_true\")\n    args = ap.parse_args()\n\n    t_start = time.perf_counter()\n    cfg = scenario_config(args.clients)\n    run = Run(args.out)\n    log = lambda m: print(m, flush=True)                                   # noqa: E731\n\n    ngpu = torch.cuda.device_count()\n    assert ngpu >= 1, \"no CUDA device\"\n    log(f\"CUDA devices: {[torch.cuda.get_device_name(i) for i in range(ngpu)]}\")\n\n    arch = Path(args.arch)\n    schema = fldata.Schema(arch / \"meta.json\", arch / \"scaler.json\")\n    train_root, test_base = resolve_paths(args.clients, args)\n    log(f\"train root: {train_root}\\ntest root : {test_base}\")\n\n    # ---- W&B first: a missing secret must cost seconds, not the whole prepack.\n    wb = None\n    if not args.no_wandb:\n        import wandb\n        wb = wandb.init(entity=args.wandb_entity, project=args.wandb_project,\n                        id=cfg[\"run_name\"], name=cfg[\"run_name\"], resume=\"allow\",\n                        mode=\"online\", dir=str(run.root),\n                        config={**cfg, \"rounds\": args.rounds, \"model\": MODEL_CFG,\n                                \"n_params\": N_PARAMS, \"spec\": \"rebuild.md Proposal A\",\n                                \"cluster_size\": afpha.CLUSTER_SIZE, \"mu_base\": afpha.MU_BASE,\n                                \"lr_max\": afpha.LR_MAX, \"lr_min\": afpha.LR_MIN,\n                                \"grad_clip\": afpha.GRAD_CLIP, \"eval_batch\": args.eval_batch,\n                                \"compile_mode_requested\": args.compile_mode})\n        wb.define_metric(\"progress/global_step\")\n        wb.define_metric(\"*\", step_metric=\"progress/global_step\")\n        (run.root / \"wandb_run.json\").write_text(json.dumps(\n            {\"entity\": wb.entity, \"project\": wb.project, \"run_id\": wb.id,\n             \"run_path\": f\"{wb.entity}/{wb.project}/{wb.id}\", \"url\": wb.url}, indent=2))\n        log(f\"W&B: {wb.url}\")\n\n    # Footers only: cheap enough to run before prepack, which is what lets\n    # --require-resume fail in seconds instead of after a multi-minute decode.\n    rows, n_test, digests = fldata.scan_counts(train_root / \"train\", test_base / \"test\")\n    assert len(rows) == args.clients, f\"{len(rows)} client partitions for {args.clients} clients\"\n    total_rows = int(rows.sum())\n    steps_round = fldata.steps_per_round(rows, cfg[\"batch\"])\n    log(f\"{args.clients} clients, {total_rows:,} rows, {n_test:,} test rows, \"\n        f\"{steps_round:,} optimizer steps/round\")\n\n    clusters = afpha.build_clusters(args.clients)\n    (run.root / \"clusters.json\").write_text(json.dumps(clusters, indent=2))\n\n    # reconstruction package: weights alone do not rebuild a model\n    shutil.copy(Path(__file__).parent / \"dagsnet.py\", run.root / \"model_source\" / \"dagsnet.py\")\n    for f in (\"meta.json\", \"scaler.json\"):\n        shutil.copy(arch / f, run.root / \"model_source\" / f)\n    (run.root / \"model_source\" / \"model_config.json\").write_text(json.dumps(\n        {\"cfg\": MODEL_CFG, \"n_features\": N_FEATURES, \"n_params\": N_PARAMS,\n         \"load\": \"build_dagsnet(); load_state_dict(torch.load(w, weights_only=True), strict=True)\"},\n        indent=2))\n\n    ref = build_dagsnet()\n    template = flatpack.Template(ref)\n    # Everything that changes what the run means. Leaving a constant out means a resume\n    # can silently continue under different mathematics.\n    fingerprint = hashlib.sha256(json.dumps(\n        {\"model\": MODEL_CFG, \"schema\": schema.fingerprint(), \"clients\": args.clients,\n         \"batch\": cfg[\"batch\"], \"rounds\": args.rounds, \"clusters\": clusters,\n         \"spec\": \"proposal-A\",\n         \"mu_base\": afpha.MU_BASE, \"mu_eps\": afpha.MU_EPS,\n         \"lr_max\": afpha.LR_MAX, \"lr_min\": afpha.LR_MIN, \"clip\": afpha.GRAD_CLIP,\n         \"adam\": [list(afpha.ADAM_BETAS), afpha.ADAM_EPS, afpha.ADAM_WEIGHT_DECAY],\n         \"cluster_size\": afpha.CLUSTER_SIZE, \"cluster_seed\": afpha.CLUSTER_SEED,\n         \"shuffle_seed\": afpha.SHUFFLE_SEED,\n         \"rows_per_client\": rows.tolist(), \"n_test\": n_test,\n         # Content identity, not just shape: without it a resume can continue onto data\n         # that was rewritten in place while keeping the same row counts.\n         \"data_digests\": digests,\n         }, sort_keys=True).encode()).hexdigest()[:16]\n\n    # ---- resume: import a prior session's output first, then read the marker\n    imported = import_previous(run, args.resume_search, fingerprint, log,\n                               explicit=args.resume_source)\n    start_round = run.last_complete_round()\n    if args.require_resume:\n        assert start_round, (\n            \"--require-resume was set but no completed round with fingerprint \"\n            f\"{fingerprint} was found under {list(args.resume_search)}\")\n    mus = afpha.initial_mu(args.clients)\n    if start_round:\n        # mu applied in round start_round+1 was derived from round start_round's drift\n        state = torch.load(run.root / \"resume\" / f\"round_{start_round:03d}.pt\",\n                           weights_only=True)\n        assert state[\"round\"] == start_round, \\\n            f\"resume state round {state['round']} != last complete {start_round}\"\n        assert state[\"fingerprint\"] == fingerprint, \"fingerprint changed; refusing to resume\"\n        mus = {int(k): float(v) for k, v in state[\"mu\"].items()}\n        gf, gi = template.flatten(torch.load(\n            run.root / \"weights\" / f\"round_{start_round:03d}.pt\", weights_only=True))\n        log(f\"resuming after round {start_round}\")\n    else:\n        torch.manual_seed(afpha.CLUSTER_SEED)\n        gf, gi = template.flatten(build_dagsnet().state_dict())\n        if run.history.exists():\n            run.history.unlink()\n    paths, spans = prepack(train_root, test_base, schema, args.scratch, rows,\n                           n_test, digests, log)\n    assert len(spans) == args.clients\n\n    (run.root / \"config.json\").write_text(json.dumps(\n        {**cfg, \"rounds\": args.rounds, \"fingerprint\": fingerprint,\n         \"train_root\": str(train_root), \"test_root\": str(test_base),\n         \"rows_per_client\": rows.tolist(), \"steps_per_round\": steps_round,\n         \"n_test\": n_test, \"eval_batch\": args.eval_batch, \"imported_rounds\": imported,\n         \"max_skip_pct\": args.max_skip_pct,\n         \"compile_mode_requested\": args.compile_mode, \"max_hours\": args.max_hours,\n         \"schema_fingerprint\": schema.fingerprint(), \"clusters\": clusters,\n         \"data_digests\": digests}, indent=2))\n\n    if start_round == 0:\n        np.save(run.root / \"preds\" / \"y_true.npy\", np.load(paths[\"test_y\"]))\n\n    # ---- workers\n    ctx = mp.get_context(\"spawn\")\n    devices = args.worker_devices or list(range(min(ngpu, 2)))\n    nw = len(devices)\n    bounds = [round(i * n_test / nw) for i in range(nw + 1)]\n    wcfg = {\"batch\": cfg[\"batch\"], \"compile_mode\": args.compile_mode,\n            \"num_classes\": schema.num_classes, \"eval_batch\": args.eval_batch,\n            \"spans\": spans}\n    task_q, res_q = ctx.Queue(), ctx.Queue()\n    ctrl = [ctx.Queue() for _ in range(nw)]\n    procs = [ctx.Process(target=worker_main,\n                         args=(r, devices[r], wcfg, paths, (bounds[r], bounds[r + 1]),\n                               ctrl[r], task_q, res_q),\n                         daemon=True) for r in range(nw)]\n    for p in procs:\n        p.start()\n\n    ready = {}\n    while len(ready) < nw:\n        kind, rank, _, info = _get(res_q, procs)\n        assert kind == \"ready\", f\"worker {rank} failed before ready: {info}\"\n        ready[rank] = info\n    log(f\"workers ready: {json.dumps(ready)}\")\n    if wb:\n        wb.config.update({\"workers\": ready}, allow_val_change=True)\n\n    order = sorted(range(args.clients), key=lambda c: -int(rows[c]))    # longest-first\n    global_step = start_round * steps_round\n    deadline = t_start + args.max_hours * 3600\n    worst_round = 0.0\n\n    try:\n        for rnd in range(start_round + 1, args.rounds + 1):\n            t_round = time.perf_counter()\n            lr = afpha.lr_for_round(rnd, args.rounds)\n            for q in ctrl:\n                q.put((\"round\", rnd, lr, mus, gf, gi))\n            for cid in order:\n                task_q.put(cid)\n            for _ in range(nw):\n                task_q.put(None)\n\n            states, stats, done, seen = {}, {}, 0, 0\n            while done < nw:\n                kind, rank, payload, info = _get(res_q, procs)\n                if kind == \"round_done\":\n                    done += 1\n                    continue\n                assert kind == \"client\", f\"unexpected {kind} during training: {info}\"\n                _check_client(info, args.max_skip_pct, args.max_skip_abs)\n                states[info[\"client_id\"]] = payload            # (float vec, int vec)\n                stats[info[\"client_id\"]] = info\n                seen += 1\n                global_step += info[\"steps\"]\n                if seen % args.heartbeat_clients == 0 or seen == args.clients:\n                    rec = {\"round\": rnd, \"clients_done\": seen, \"of\": args.clients,\n                           \"lr\": lr, \"ce_mean\": info[\"ce_mean\"],\n                           \"grad_norm\": info[\"grad_norm_mean\"], \"skipped\": info[\"skipped\"],\n                           \"elapsed_s\": round(time.perf_counter() - t_round, 1)}\n                    run.emit(rec, wb, {\"progress/round\": rnd, \"progress/global_step\": global_step,\n                                       \"progress/clients_done\": seen,\n                                       \"train/loss\": info[\"ce_mean\"],\n                                       \"train/grad_norm\": info[\"grad_norm_mean\"],\n                                       \"train/lr\": lr,\n                                       \"train/global_batch\": cfg[\"batch\"],\n                                       \"train/skip_pct\": 100.0 * info[\"skipped\"]\n                                       / max(info[\"steps\"], 1)}, global_step)\n            assert len(states) == args.clients, f\"only {len(states)} clients returned\"\n            t_train = time.perf_counter() - t_round\n\n            t_agg = time.perf_counter()\n            client_rows = {c: stats[c][\"rows\"] for c in stats}\n            gf, gi, cluster_rows = afpha.hierarchical_aggregate(\n                {c: v[0] for c, v in states.items()}, {c: v[1] for c, v in states.items()},\n                client_rows, clusters)\n            drifts = {c: afpha.param_drift(states[c][0], gf, template.n_params) for c in stats}\n            mu_used = {c: stats[c][\"mu\"] for c in stats}\n            mus = afpha.next_mu(drifts, client_rows)          # applies from the NEXT round\n            t_agg = time.perf_counter() - t_agg\n\n            t_eval = time.perf_counter()\n            for q in ctrl:\n                q.put((\"eval\", rnd, gf, gi))\n            cm = np.zeros((schema.num_classes, schema.num_classes), dtype=np.int64)\n            shard_preds = {}\n            got = 0\n            while got < 2 * nw:\n                kind, rank, payload, info = _get(res_q, procs)\n                if kind == \"eval\":\n                    cm += payload\n                elif kind == \"preds\":\n                    shard_preds[rank] = payload\n                else:\n                    raise AssertionError(f\"unexpected {kind} during eval: {info}\")\n                got += 1\n            preds = np.concatenate([shard_preds[r] for r in range(nw)])\n            assert len(preds) == n_test and int(cm.sum()) == n_test, \\\n                f\"test coverage {len(preds)}/{int(cm.sum())} != {n_test}\"\n            t_eval = time.perf_counter() - t_eval\n\n            timing = {\"train_s\": t_train, \"aggregate_s\": t_agg, \"eval_s\": t_eval,\n                      \"round_s\": time.perf_counter() - t_round}   # train+aggregate+eval\n            m = run.commit(rnd, template.unflatten(gf, gi), cm, preds,\n                           [stats[c] for c in sorted(stats)], cluster_rows,\n                           {str(k): v for k, v in mus.items()},\n                           {str(k): v for k, v in drifts.items()}, timing, schema,\n                           resume_state={\"round\": rnd, \"fingerprint\": fingerprint,\n                                         \"mu\": {str(k): float(v) for k, v in mus.items()},\n                                         \"global_step\": int(global_step)})\n\n            # Budget on the worst round seen, not the last one: round 1 pays compile and\n            # is slow, later rounds vary, and an optimistic estimate is the one that gets\n            # the session hard-killed mid-commit. Unlike timing[\"round_s\"], this includes\n            # the commit that follows evaluation.\n            commit_s = time.perf_counter() - t_round - timing[\"round_s\"]\n            worst_round = max(worst_round, time.perf_counter() - t_round)\n\n            run.emit({\"round\": rnd, \"completed\": True, **{k: round(m[k], 6) for k in METRIC_KEYS},\n                      **{k: round(v, 1) for k, v in timing.items()},\n                      \"commit_s\": round(commit_s, 1),\n                      \"worst_round_s\": round(worst_round, 1)},\n                     wb, {\"progress/completed_round\": rnd,\n                          \"progress/global_step\": global_step,\n                          \"round/duration_s\": timing[\"round_s\"],\n                          \"round/train_s\": t_train, \"round/eval_s\": t_eval,\n                          \"round/aggregate_s\": t_agg,\n                          \"round/commit_s\": commit_s,\n                          \"round/worst_round_s\": worst_round,\n                          \"round/max_skip_pct_seen\": max(v[\"skip_pct\"] for v in stats.values()),\n                          \"round/lr\": lr,\n                          \"round/mu_mean\": float(np.mean(list(mu_used.values()))),\n                          \"round/mu_next_mean\": float(np.mean(list(mus.values()))),\n                          \"round/drift_mean\": float(np.mean(list(drifts.values()))),\n                          **{f\"eval/{k}\": m[k] for k in METRIC_KEYS}}, global_step)\n\n            remaining = args.rounds - rnd\n            if remaining and time.perf_counter() + worst_round * 1.15 > deadline:\n                log(f\"stopping cleanly after round {rnd}: another round at the worst \"\n                    f\"observed {worst_round / 60:.1f} min would pass the {args.max_hours} h \"\n                    \"budget. Resume attaches this output.\")\n                break\n    finally:\n        for q in ctrl:\n            q.put((\"stop\",))\n        for p in procs:\n            p.join(timeout=120)\n        if wb:\n            wb.finish()\n    log(f\"done in {(time.perf_counter() - t_start) / 3600:.2f} h; \"\n        f\"last complete round {run.last_complete_round()}\")\n\n\ndef _check_client(info, max_skip_pct, max_skip_abs):\n    \"\"\"Reject a client update before it can reach the aggregator.\n\n    Finite weights are not enough. A client whose AMP steps were all skipped returns the\n    global weights unchanged and finite, and averaging it silently reports participation\n    that did not happen. Failing the round keeps every committed round intact and lets a\n    resume retry it; dropping the client instead would change the participation rate\n    mid-run, which the specification fixes at 100%.\n\n    The loss and gradient means arrive already restricted to applied steps, so a\n    non-finite value here means the steps that actually updated the model diverged --\n    not that the scaler skipped one while calibrating.\n    \"\"\"\n    cid = info[\"client_id\"]\n    problems = []\n    if not info[\"finite_weights\"]:\n        problems.append(\"non-finite weights\")\n    if not info[\"finite_loss\"]:\n        problems.append(f\"non-finite mean loss ({info['ce_mean']})\")\n    if not info[\"finite_grad\"]:\n        problems.append(f\"non-finite mean gradient norm ({info['grad_norm_mean']})\")\n    if info[\"steps\"] and info[\"skipped\"] >= info[\"steps\"]:\n        problems.append(f\"every one of {info['steps']} optimizer steps was skipped\")\n    else:\n        allowed = max(max_skip_abs, math.ceil(max_skip_pct / 100 * info[\"steps\"]))\n        if info[\"skipped\"] > allowed:\n            problems.append(f\"skipped {info['skipped']}/{info['steps']} steps \"\n                            f\"({info['skip_pct']:.2f}%), above the allowance of {allowed} \"\n                            f\"(max({max_skip_abs} warm-up, {max_skip_pct}%))\")\n    if problems:\n        raise RuntimeError(\n            f\"client {cid} update rejected: {'; '.join(problems)}. \"\n            f\"rows={info['rows']} steps={info['steps']} skipped={info['skipped']} \"\n            f\"lr={info['lr']} mu={info['mu']} device={info['device']}. \"\n            \"Rounds already committed are intact; resume retries this round.\")\n\n\ndef _get(q, procs=None, timeout=7200, poll=20.0):\n    \"\"\"Block for a worker message, but notice a dead worker while waiting.\n\n    A worker killed by the OS (OOM most likely) never puts anything on the queue, so a\n    single long `get` would sit for the whole timeout. Poll in short slices and check\n    liveness between them.\n    \"\"\"\n    deadline = time.perf_counter() + timeout\n    while True:\n        try:\n            kind, rank, payload, info = q.get(timeout=poll)\n            break\n        except queue.Empty:\n            if procs:\n                dead = [i for i, pr in enumerate(procs)\n                        if not pr.is_alive() and pr.exitcode not in (None, 0)]\n                if dead:\n                    codes = {i: procs[i].exitcode for i in dead}\n                    raise RuntimeError(\n                        f\"worker(s) {codes} exited without reporting. A negative code is a \"\n                        \"signal (-9 is the OOM killer). Committed rounds are intact.\")\n            if time.perf_counter() > deadline:\n                raise TimeoutError(f\"no worker message for {timeout}s\")\n    if kind == \"error\":\n        raise RuntimeError(f\"worker {rank} crashed:\\n{info['traceback']}\")\n    return kind, rank, payload, info\n\n\nif __name__ == \"__main__\":\n    os.environ.setdefault(\"OMP_NUM_THREADS\", \"1\")\n    main()\n",
 "meta.json": "{\n  \"class_names\": [\n    \"benign\",\n    \"accelerationMultiplication\",\n    \"constantPositionOffset\",\n    \"constantSpeedOffset\",\n    \"dataReplay\",\n    \"dosAttack\",\n    \"feignedBraking\",\n    \"positionMirroring\",\n    \"randomPositionOffset\",\n    \"randomSpeedOffset\",\n    \"reversedHeading\",\n    \"suddenConstantSpeed\",\n    \"suddenStop\",\n    \"timeDelayAttack\",\n    \"trafficCongestionSybil\",\n    \"zeroSpeedReport\"\n  ],\n  \"num_classes\": 16,\n  \"feature_cols\": [\n    \"f_rcv_pos_noise_x\",\n    \"f_rcv_pos_noise_y\",\n    \"f_rcv_spd\",\n    \"f_rcv_spd_noise\",\n    \"f_rcv_acl\",\n    \"f_rcv_acl_noise\",\n    \"f_rcv_hed_noise\",\n    \"f_snd_pos_noise_x\",\n    \"f_snd_pos_noise_y\",\n    \"f_snd_spd\",\n    \"f_snd_spd_noise\",\n    \"f_snd_acl\",\n    \"f_snd_acl_noise\",\n    \"f_snd_hed_noise\",\n    \"f_snd_dist_road_edge\",\n    \"f_rcv_x_rel\",\n    \"f_rcv_y_rel\",\n    \"f_snd_x_rel\",\n    \"f_snd_y_rel\",\n    \"f_delay_s\",\n    \"f_dx\",\n    \"f_dy\",\n    \"f_dist\",\n    \"f_bearing_sin\",\n    \"f_bearing_cos\",\n    \"f_rcv_hed_sin\",\n    \"f_rcv_hed_cos\",\n    \"f_snd_hed_sin\",\n    \"f_snd_hed_cos\",\n    \"f_hed_diff_cos\",\n    \"f_rcv_vx\",\n    \"f_rcv_vy\",\n    \"f_snd_vx\",\n    \"f_snd_vy\",\n    \"f_rel_speed\",\n    \"f_closing_speed\",\n    \"f_spd_diff\",\n    \"f_rcv_noise_mag\",\n    \"f_snd_noise_mag\",\n    \"f_first_in_session\",\n    \"f_sess_idx\",\n    \"f_sess_dt\",\n    \"f_sess_dt_send\",\n    \"f_sess_dt_skew\",\n    \"f_sess_dpos\",\n    \"f_sess_implied_spd\",\n    \"f_sess_spd_residual\",\n    \"f_sess_dspd\",\n    \"f_sess_acl_residual\",\n    \"f_sess_dhed\",\n    \"f_sess_dmsgid\",\n    \"f_sess_ddist\",\n    \"f_sess_ddre\",\n    \"f_sess_pos_pred_err\",\n    \"f_alias_age_s\",\n    \"f_rx_rate_1s\",\n    \"f_rx_rate_5s\",\n    \"f_sender_rate_1s\",\n    \"f_sender_rate_5s\",\n    \"f_sender_share_5s\",\n    \"f_rcv_profile_normal\",\n    \"f_rcv_profile_cautious\",\n    \"f_rcv_profile_aggressive\",\n    \"f_snd_profile_normal\",\n    \"f_snd_profile_cautious\",\n    \"f_snd_profile_aggressive\"\n  ],\n  \"dropped_cols\": [\n    \"receiver_id\",\n    \"sender_id\",\n    \"sender_alias\",\n    \"message_id\",\n    \"rcv_time_ns\",\n    \"send_time_ns\",\n    \"orig_split\",\n    \"source_run\",\n    \"t_rel_s\",\n    \"label\",\n    \"rcv_pos_x\",\n    \"rcv_pos_y\",\n    \"snd_pos_x\",\n    \"snd_pos_y\",\n    \"rcv_hed\",\n    \"snd_hed\",\n    \"rcv_profile\",\n    \"snd_profile\",\n    \"snd_dist_road_edge\",\n    \"attack_type\",\n    \"scenario\"\n  ],\n  \"n_train\": 43045415,\n  \"n_test\": 10761343,\n  \"train_counts\": [\n    9564547,\n    629963,\n    1769893,\n    1732706,\n    1901664,\n    6296361,\n    474418,\n    1868199,\n    1881029,\n    2168752,\n    1079992,\n    231028,\n    845778,\n    1962298,\n    9573355,\n    1065432\n  ],\n  \"test_counts\": [\n    2391136,\n    157490,\n    442475,\n    433177,\n    475410,\n    1574090,\n    118605,\n    467049,\n    470279,\n    542163,\n    269999,\n    57757,\n    211445,\n    490574,\n    2393335,\n    266359\n  ],\n  \"scaler\": {\n    \"applied_to\": \"test_only\",\n    \"fitted_on\": \"train_only\",\n    \"ddof\": 0\n  },\n  \"paths\": {\n    \"train_X\": \"/kaggle/temp/veremi_cache/train_X.f16.npy\",\n    \"train_y\": \"/kaggle/temp/veremi_cache/train_y.i8.npy\",\n    \"test_X\": \"/kaggle/temp/veremi_cache/test_X.f16.npy\",\n    \"test_y\": \"/kaggle/temp/veremi_cache/test_y.i8.npy\"\n  }\n}",
 "scaler.json": "{\n  \"features\": {\n    \"f_alias_age_s\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 306.0006103515625,\n      \"mean\": 39.61483940962532,\n      \"min\": 0.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 55.74762867161773,\n      \"std_used\": 55.74762867161773\n    },\n    \"f_bearing_cos\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1.0,\n      \"mean\": 0.00019984038923879233,\n      \"min\": -1.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.8144757171102179,\n      \"std_used\": 0.8144757171102179\n    },\n    \"f_bearing_sin\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1.0,\n      \"mean\": -0.00574563504835482,\n      \"min\": -1.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.5801691598106258,\n      \"std_used\": 0.5801691598106258\n    },\n    \"f_closing_speed\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 117.9191665649414,\n      \"mean\": 0.3010788730381897,\n      \"min\": -117.74347686767578,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 14.41548400050646,\n      \"std_used\": 14.41548400050646\n    },\n    \"f_delay_s\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 0.006609446834772825,\n      \"mean\": 0.001563238040346154,\n      \"min\": 0.0009440850117243826,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.00043212129606767314,\n      \"std_used\": 0.00043212129606767314\n    },\n    \"f_dist\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 419.47393798828125,\n      \"mean\": 154.36084475389623,\n      \"min\": 0.048602327704429626,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 91.25424806815055,\n      \"std_used\": 91.25424806815055\n    },\n    \"f_dx\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 392.0463562011719,\n      \"mean\": -1.6461432503972113,\n      \"min\": -389.7549133300781,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 101.21425021648632,\n      \"std_used\": 101.21425021648632\n    },\n    \"f_dy\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 395.3658752441406,\n      \"mean\": 1.111092186701751,\n      \"min\": -395.7381286621094,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 148.00790328431185,\n      \"std_used\": 148.00790328431185\n    },\n    \"f_first_in_session\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1.0,\n      \"mean\": 0.240895528594625,\n      \"min\": 0.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.42762702545295384,\n      \"std_used\": 0.42762702545295384\n    },\n    \"f_hed_diff_cos\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1.0,\n      \"mean\": 0.17828701239780986,\n      \"min\": -1.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.7982453384120445,\n      \"std_used\": 0.7982453384120445\n    },\n    \"f_rcv_acl\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 3.11494779586792,\n      \"mean\": -0.04328791153446996,\n      \"min\": -9.059639930725098,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 1.1755899899721087,\n      \"std_used\": 1.1755899899721087\n    },\n    \"f_rcv_acl_noise\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 0.24137607216835022,\n      \"mean\": -5.348051440966623e-08,\n      \"min\": -0.22659999132156372,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.0004231437184362911,\n      \"std_used\": 0.0004231437184362911\n    },\n    \"f_rcv_hed_cos\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1.0,\n      \"mean\": -0.03482181330162422,\n      \"min\": -1.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.8074230139968959,\n      \"std_used\": 0.8074230139968959\n    },\n    \"f_rcv_hed_noise\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 19.96363067626953,\n      \"mean\": -0.08399356547323075,\n      \"min\": -19.983116149902344,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 8.468671104180709,\n      \"std_used\": 8.468671104180709\n    },\n    \"f_rcv_hed_sin\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1.0,\n      \"mean\": -0.006941082932810775,\n      \"min\": -1.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.5889035057073401,\n      \"std_used\": 0.5889035057073401\n    },\n    \"f_rcv_noise_mag\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 7.596007347106934,\n      \"mean\": 3.6981504765120574,\n      \"min\": 0.0427437424659729,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 1.3621844035831625,\n      \"std_used\": 1.3621844035831625\n    },\n    \"f_rcv_pos_noise_x\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 5.645547389984131,\n      \"mean\": -0.3222908417675181,\n      \"min\": -5.713721752166748,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 2.7170260562784825,\n      \"std_used\": 2.7170260562784825\n    },\n    \"f_rcv_pos_noise_y\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 5.704358100891113,\n      \"mean\": 0.052102859259389595,\n      \"min\": -5.774186134338379,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 2.83602655305172,\n      \"std_used\": 2.83602655305172\n    },\n    \"f_rcv_profile_aggressive\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1.0,\n      \"mean\": 0.07342605478423196,\n      \"min\": 0.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.26083456301467206,\n      \"std_used\": 0.26083456301467206\n    },\n    \"f_rcv_profile_cautious\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1.0,\n      \"mean\": 0.098501431569425,\n      \"min\": 0.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.2979914420720818,\n      \"std_used\": 0.2979914420720818\n    },\n    \"f_rcv_profile_normal\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1.0,\n      \"mean\": 0.828072513646343,\n      \"min\": 0.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.3773174072180742,\n      \"std_used\": 0.3773174072180742\n    },\n    \"f_rcv_spd\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 55.570858001708984,\n      \"mean\": 7.573656458085321,\n      \"min\": 0.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 9.885612124282533,\n      \"std_used\": 9.885612124282533\n    },\n    \"f_rcv_spd_noise\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 0.03383542597293854,\n      \"mean\": 7.136993612772818e-05,\n      \"min\": -0.026916449889540672,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.002071984991087657,\n      \"std_used\": 0.002071984991087657\n    },\n    \"f_rcv_vx\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 55.54975128173828,\n      \"mean\": 0.005219854374029014,\n      \"min\": -55.34329605102539,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 6.357658706003342,\n      \"std_used\": 6.357658706003342\n    },\n    \"f_rcv_vy\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 55.54323959350586,\n      \"mean\": -0.5175092532246992,\n      \"min\": -55.5601921081543,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 10.695696893708499,\n      \"std_used\": 10.695696893708499\n    },\n    \"f_rcv_x_rel\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 3712.850341796875,\n      \"mean\": 1132.7658840829756,\n      \"min\": 107.13914489746094,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 858.0027430847754,\n      \"std_used\": 858.0027430847754\n    },\n    \"f_rcv_y_rel\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 5329.70703125,\n      \"mean\": 2957.053123778659,\n      \"min\": 728.54541015625,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 1489.3207087357348,\n      \"std_used\": 1489.3207087357348\n    },\n    \"f_rel_speed\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 117.96372985839844,\n      \"mean\": 10.998055250459968,\n      \"min\": 0.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 13.28465701726787,\n      \"std_used\": 13.28465701726787\n    },\n    \"f_rx_rate_1s\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 294.0,\n      \"mean\": 75.11689370401005,\n      \"min\": 1.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 49.41655629713506,\n      \"std_used\": 49.41655629713506\n    },\n    \"f_rx_rate_5s\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1416.0,\n      \"mean\": 369.56791560727197,\n      \"min\": 1.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 247.46265458181946,\n      \"std_used\": 247.46265458181946\n    },\n    \"f_sender_rate_1s\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 5.0,\n      \"mean\": 1.6916893007071716,\n      \"min\": 1.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 1.0020295633305791,\n      \"std_used\": 1.0020295633305791\n    },\n    \"f_sender_rate_5s\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 21.0,\n      \"mean\": 5.686044239554898,\n      \"min\": 1.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 4.841997296508535,\n      \"std_used\": 4.841997296508535\n    },\n    \"f_sender_share_5s\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1.0,\n      \"mean\": 0.0422330102040291,\n      \"min\": 0.0007062146905809641,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.1078109926968647,\n      \"std_used\": 0.1078109926968647\n    },\n    \"f_sess_acl_residual\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 55.66212463378906,\n      \"mean\": 0.05029493947361385,\n      \"min\": -55.554805755615234,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 1.9460356785187678,\n      \"std_used\": 1.9460356785187678\n    },\n    \"f_sess_ddist\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 371.71905517578125,\n      \"mean\": -0.19965569562884664,\n      \"min\": -379.4576110839844,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 23.87013297074142,\n      \"std_used\": 23.87013297074142\n    },\n    \"f_sess_ddre\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 98.17662811279297,\n      \"mean\": 0.002369458930881372,\n      \"min\": -95.54478454589844,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 5.884758745778502,\n      \"std_used\": 5.884758745778502\n    },\n    \"f_sess_dhed\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 179.9944305419922,\n      \"mean\": 0.18283055556685665,\n      \"min\": -179.99771118164062,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 10.087760396119801,\n      \"std_used\": 10.087760396119801\n    },\n    \"f_sess_dmsgid\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 2000000.0,\n      \"mean\": 2386.8923205642227,\n      \"min\": -2000000.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 619913.8106454061,\n      \"std_used\": 619913.8106454061\n    },\n    \"f_sess_dpos\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1791.1820068359375,\n      \"mean\": 10.116825837410033,\n      \"min\": 0.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 30.873938878136784,\n      \"std_used\": 30.873938878136784\n    },\n    \"f_sess_dspd\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 55.538658142089844,\n      \"mean\": 0.0012256514996186578,\n      \"min\": -58.1681022644043,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 2.002065282597647,\n      \"std_used\": 2.002065282597647\n    },\n    \"f_sess_dt\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 250.000244140625,\n      \"mean\": 0.7099338794911559,\n      \"min\": 0.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 1.3999639308222824,\n      \"std_used\": 1.3999639308222824\n    },\n    \"f_sess_dt_send\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 250.0,\n      \"mean\": 0.7099338352740506,\n      \"min\": 0.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 1.3999635488396631,\n      \"std_used\": 1.3999635488396631\n    },\n    \"f_sess_dt_skew\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 0.005492310971021652,\n      \"mean\": 4.4553514706567495e-08,\n      \"min\": -0.005396704189479351,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.0004235553094810664,\n      \"std_used\": 0.0004235553094810664\n    },\n    \"f_sess_idx\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1196.0,\n      \"mean\": 54.580317671463966,\n      \"min\": 0.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 100.50522208930417,\n      \"std_used\": 100.50522208930417\n    },\n    \"f_sess_implied_spd\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 824.9472045898438,\n      \"mean\": 9.884142022731766,\n      \"min\": 0.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 29.011264711827106,\n      \"std_used\": 29.011264711827106\n    },\n    \"f_sess_pos_pred_err\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 9388.2822265625,\n      \"mean\": 7.435471622796253,\n      \"min\": 0.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 32.107792229575146,\n      \"std_used\": 32.107792229575146\n    },\n    \"f_sess_spd_residual\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 824.2792358398438,\n      \"mean\": 4.546495165552884,\n      \"min\": -56.79338455200195,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 27.721740247255553,\n      \"std_used\": 27.721740247255553\n    },\n    \"f_snd_acl\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 12.304226875305176,\n      \"mean\": -0.05753384587639149,\n      \"min\": -32.78059768676758,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 1.3625369341563571,\n      \"std_used\": 1.3625369341563571\n    },\n    \"f_snd_acl_noise\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 0.005553900729864836,\n      \"mean\": 3.338423920901895e-07,\n      \"min\": -0.00789023656398058,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 9.636459266028168e-05,\n      \"std_used\": 9.636459266028168e-05\n    },\n    \"f_snd_dist_road_edge\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 14.791890144348145,\n      \"mean\": 1.4073348374682044,\n      \"min\": -98.64256286621094,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 9.526795046650534,\n      \"std_used\": 9.526795046650534\n    },\n    \"f_snd_hed_cos\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1.0,\n      \"mean\": -0.029387492535223575,\n      \"min\": -1.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.8089665948480084,\n      \"std_used\": 0.8089665948480084\n    },\n    \"f_snd_hed_noise\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 21.836334228515625,\n      \"mean\": 0.027575087556552292,\n      \"min\": -21.88494110107422,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 8.336442164831045,\n      \"std_used\": 8.336442164831045\n    },\n    \"f_snd_hed_sin\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1.0,\n      \"mean\": -0.01701571351177838,\n      \"min\": -1.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.586872975311675,\n      \"std_used\": 0.586872975311675\n    },\n    \"f_snd_noise_mag\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 7.906435489654541,\n      \"mean\": 3.7884973059485105,\n      \"min\": 0.04344702884554863,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 1.3730877268923125,\n      \"std_used\": 1.3730877268923125\n    },\n    \"f_snd_pos_noise_x\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 5.930288314819336,\n      \"mean\": -0.12869869726950448,\n      \"min\": -5.876334190368652,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 2.782897997335108,\n      \"std_used\": 2.782897997335108\n    },\n    \"f_snd_pos_noise_y\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 5.897462844848633,\n      \"mean\": -0.22331545685848006,\n      \"min\": -6.031744003295898,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 2.9029514871785316,\n      \"std_used\": 2.9029514871785316\n    },\n    \"f_snd_profile_aggressive\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1.0,\n      \"mean\": 0.06636293319509175,\n      \"min\": 0.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.24891543602765084,\n      \"std_used\": 0.24891543602765084\n    },\n    \"f_snd_profile_cautious\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1.0,\n      \"mean\": 0.0975302479950536,\n      \"min\": 0.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.2966784433036498,\n      \"std_used\": 0.2966784433036498\n    },\n    \"f_snd_profile_normal\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 1.0,\n      \"mean\": 0.8361068188098547,\n      \"min\": 0.0,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.37017861411691455,\n      \"std_used\": 0.37017861411691455\n    },\n    \"f_snd_spd\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 62.49550247192383,\n      \"mean\": 7.213736120885268,\n      \"min\": -6.99880838394165,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 9.986729178675164,\n      \"std_used\": 9.986729178675164\n    },\n    \"f_snd_spd_noise\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 0.03383542597293854,\n      \"mean\": 6.525150774013438e-05,\n      \"min\": -0.026916449889540672,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 0.002040353436698,\n      \"std_used\": 0.002040353436698\n    },\n    \"f_snd_vx\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 62.356441497802734,\n      \"mean\": -0.0008791727307567289,\n      \"min\": -60.52200698852539,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 6.220102066248368,\n      \"std_used\": 6.220102066248368\n    },\n    \"f_snd_vy\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 62.35265350341797,\n      \"mean\": -0.3821577119997693,\n      \"min\": -62.49283218383789,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 10.62718370968904,\n      \"std_used\": 10.62718370968904\n    },\n    \"f_snd_x_rel\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 3773.819580078125,\n      \"mean\": 1131.1197406511137,\n      \"min\": 68.19474792480469,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 856.0188021490834,\n      \"std_used\": 856.0188021490834\n    },\n    \"f_snd_y_rel\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 5387.63134765625,\n      \"mean\": 2958.1642159303824,\n      \"min\": 664.7039184570312,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 1487.6271502605048,\n      \"std_used\": 1487.6271502605048\n    },\n    \"f_spd_diff\": {\n      \"degenerate\": false,\n      \"fill_value\": 0.0,\n      \"max\": 62.475303649902344,\n      \"mean\": -0.35992033717961164,\n      \"min\": -62.53355026245117,\n      \"missing_frac\": 0.0,\n      \"n_missing\": 0,\n      \"std\": 12.22382527023506,\n      \"std_used\": 12.22382527023506\n    }\n  },\n  \"imputation\": {\n    \"applies_to\": \"non-finite feature values\",\n    \"method\": \"constant\",\n    \"stage\": \"Phase 2, before the 80/20 time cut - both splits arrive here imputed\",\n    \"value\": 0.0\n  },\n  \"n_train_rows\": 43045415,\n  \"standardization\": {\n    \"fit_scope\": \"centralized/train\",\n    \"method\": \"StandardScaler\",\n    \"variance_ddof\": 0\n  }\n}"
}

In [ ]:
SRC_DIR = "/kaggle/working/src"
os.makedirs(SRC_DIR, exist_ok=True)
for name, text in MODULES.items():
    with open(os.path.join(SRC_DIR, name), "w") as fh:
        fh.write(text)
sys.path.insert(0, SRC_DIR)
import dagsnet, metrics                                   # noqa: E402
_meta = json.load(open(f"{SRC_DIR}/meta.json"))
_scaler = json.load(open(f"{SRC_DIR}/scaler.json"))["features"]
assert len(_meta["feature_cols"]) == 66 and _meta["num_classes"] == 16
assert set(_meta["feature_cols"]) == set(_scaler), "scaler and feature order disagree"
m = dagsnet.build_dagsnet()
n = sum(p.numel() for p in m.parameters())
b = sum(x.numel() for x in m.buffers())
print(f"DAGSNet rebuilt: {n:,} parameters, {b:,} buffer elements, "
      f"forward {tuple(m(torch.zeros(4, 66)).shape)}")
assert n == dagsnet.N_PARAMS and b == 3295
print("metric keys:", metrics.METRIC_KEYS)

## 4. Backend micro-benchmark (synthetic; proves Inductor works on sm75)

In [ ]:
# Micro-benchmark: informational, and it proves Inductor works on sm75 before the real
# round pays for it. AMP=False is measured for the record only -- production is always
# AMP=True, so it is never a candidate configuration.
import time, torch
from dagsnet import build_dagsnet
from afpha import ADAM_BETAS, ADAM_EPS, GRAD_CLIP

def bench(batch, mode, amp, steps=300, warmup=120, pool=200_000):
    dev = "cuda:0"
    torch.backends.cudnn.benchmark = True
    torch.manual_seed(0)
    X = torch.randn(pool, 66, device=dev, dtype=torch.float16)
    Y = torch.randint(0, 16, (pool,), device=dev)
    model = build_dagsnet().to(dev)
    net = model if mode == "eager" else torch.compile(
        model, mode=None if mode == "default" else mode)
    params = list(model.parameters())
    w_g = [p.detach().clone() for p in params]
    opt = torch.optim.Adam(params, lr=1e-3, betas=ADAM_BETAS, eps=ADAM_EPS, fused=True)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    crit = torch.nn.CrossEntropyLoss()
    g = torch.Generator(device=dev); g.manual_seed(1)
    ce = torch.zeros((), device=dev)

    def one():
        idx = torch.randint(0, pool, (batch,), device=dev, generator=g)
        xb = X[idx]
        if amp:
            with torch.autocast("cuda", dtype=torch.float16):
                loss = crit(net(xb), Y[idx])
            scaler.scale(loss).backward(); scaler.unscale_(opt)
        else:
            loss = crit(net(xb.float()), Y[idx]); loss.backward()
        torch._foreach_add_([p.grad for p in params],
                            torch._foreach_sub(params, w_g), alpha=0.01)
        torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
        if amp:
            scaler.step(opt); scaler.update()
        else:
            opt.step()
        opt.zero_grad(set_to_none=True)
        ce.add_(loss.detach())

    try:
        torch._dynamo.reset()
        for _ in range(warmup): one()
        torch.cuda.synchronize(); t0 = time.perf_counter()
        for _ in range(steps): one()
        torch.cuda.synchronize(); dt = (time.perf_counter() - t0) / steps
        peak = torch.cuda.max_memory_allocated(dev) / 2**30
    except Exception as exc:
        return {"batch": batch, "mode": mode, "amp": amp,
                "error": f"{type(exc).__name__}: {exc}"}
    finally:
        del model, net, opt
        torch._dynamo.reset(); torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(dev)
    return {"batch": batch, "mode": mode, "amp": amp, "candidate": bool(amp),
            "ms_per_step": round(dt * 1e3, 3), "samples_per_s": round(batch / dt),
            "peak_GiB": round(peak, 3)}

STEP = []
for batch in (512, 256):
    for mode, amp in (("eager", True), ("default", True),
                      ("reduce-overhead", True), ("reduce-overhead", False)):
        r = bench(batch, mode, amp)
        STEP.append(r); print(json.dumps(r), flush=True)

CANDIDATES = [r for r in STEP if r.get("candidate") and "ms_per_step" in r]
assert CANDIDATES, "no AMP configuration survived the micro-benchmark"

## 5. The production driver, for real

Three rounds of the actual 20-client run. This is the number that budgets the launch.

In [ ]:
# The real thing: the production driver, the real 20-client partition, both GPUs, the
# full test set. Everything the synthetic loop above cannot measure -- parquet decode,
# worker startup, compile validation, resident-VRAM fit, eval sharding, commit I/O --
# is in these numbers.
CAL_DIR = "/kaggle/working/calib"
CMD = [sys.executable, f"{SRC_DIR}/fl_train.py",
       "--clients", "20", "--out", CAL_DIR, "--scratch", "/kaggle/temp/prepack",
       "--arch", SRC_DIR, "--data-search", "/kaggle/input", "--test-search", "/kaggle/input",
       "--rounds", str(CAL_ROUNDS), "--eval-batch", "16384",
       "--compile-mode", "reduce-overhead", "--max-hours", str(CAL_MAX_HOURS),
       "--no-wandb"]
print(" ".join(CMD), flush=True)
t0 = time.time()
proc = subprocess.Popen(CMD, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
CAL_LOG = []
for line in proc.stdout:
    CAL_LOG.append(line.rstrip())
    print(line, end="", flush=True)
rc = proc.wait()
CAL_WALL = time.time() - t0
print(f"\nexit={rc} after {CAL_WALL/60:.1f} min", flush=True)
assert rc == 0, f"calibration run exited {rc}"

## 6. Measured cost, and the projection for all three scenarios

In [ ]:
# Derive the production numbers from the real run, then project the other two scenarios.
import csv, math, glob
import fldata

cal_cfg = json.load(open(f"{CAL_DIR}/config.json"))
rounds = list(csv.DictReader(open(f"{CAL_DIR}/metrics/history.csv")))
prepack = json.load(open("/kaggle/temp/prepack/prepack.json"))
workers = [json.loads(l.split("workers ready: ")[1]) for l in CAL_LOG if "workers ready: " in l]
compile_modes = sorted({w["compile_mode"] for w in workers[0].values()}) if workers else []
resident = max((w["resident_GiB"] for w in workers[0].values()), default=None) if workers else None

# Round 1 pays compile; steady state is the median of the rest when there is one.
steady = rounds[1:] or rounds
def med(vals):
    v = sorted(vals); return v[len(v) // 2]
t_train = med(float(r["t_train_s"]) for r in steady)
t_eval = med(float(r["t_eval_s"]) for r in steady)
t_round = med(float(r["t_round_s"]) for r in steady)
# history's round_s is train+aggregate+eval; commit I/O is only in the heartbeat, and
# leaving it out of the budget is how a session gets hard-killed mid-write.
beats = [json.loads(l) for l in open(f"{CAL_DIR}/heartbeat.jsonl")]
done_beats = [b for b in beats if b.get("completed")][1:] or [b for b in beats if b.get("completed")]
t_commit = med(b["commit_s"] for b in done_beats)
t_full = t_round + t_commit
steps_20 = int(cal_cfg["steps_per_round"])
# two GPUs share the round's steps
MS_PER_STEP_512 = t_train * 1000.0 * 2 / steps_20

ratio_256 = None
c512 = [r for r in CANDIDATES if r["batch"] == 512 and r["mode"] == "reduce-overhead"]
c256 = [r for r in CANDIDATES if r["batch"] == 256 and r["mode"] == "reduce-overhead"]
if c512 and c256:
    ratio_256 = c256[0]["ms_per_step"] / c512[0]["ms_per_step"]

PROJ = {}
for n in (20, 50, 100):
    try:
        root = fldata.find_root(f"{n}_client/client_stats.json", ["/kaggle/input"]) / f"{n}_client"
    except FileNotFoundError as exc:
        PROJ[n] = {"error": str(exc)[:160]}
        continue
    stats = json.load(open(root / "client_stats.json"))
    rws = [c["rows"] for c in stats["clients"]]
    batch = 512 if n in (20, 50) else 256
    steps = sum(math.ceil(r / batch) for r in rws)
    ms = MS_PER_STEP_512 * (ratio_256 if batch == 256 and ratio_256 else 1.0)
    train_h = steps * ms / 1000 / 2 * 50 / 3600
    other_h = (t_full - t_train) * 50 / 3600           # eval + aggregate + commit
    PROJ[n] = {"clients": len(rws), "rows": int(sum(rws)), "steps_per_round": steps,
               "ms_per_step": round(ms, 3),
               "train_h": round(train_h, 2), "other_h": round(other_h, 2),
               "total_h": round(train_h + other_h, 2),
               "resident_GiB": round(sum(rws) * 66 * 2 / 2**30, 2),
               "suggested_max_hours": round(train_h + other_h + 0.5, 1)}
    print(json.dumps({n: PROJ[n]}, indent=2), flush=True)

total = sum(v["total_h"] for v in PROJ.values() if "total_h" in v)
print(f"\nprojected GPU hours for all scenarios: {total:.2f}")
print("MEASURED on this session, not extrapolated from another GPU:")
print(json.dumps({"compile_modes": compile_modes, "resident_GiB_after_load": resident,
                  "train_decode_s": prepack.get("train_decode_s"),
                  "test_decode_s": prepack.get("test_decode_s"),
                  "median_train_s": round(t_train, 1), "median_eval_s": round(t_eval, 1),
                  "median_commit_s": round(t_commit, 1),
                  "median_full_round_s": round(t_full, 1),
                  "ms_per_step_batch512": round(MS_PER_STEP_512, 3)}, indent=2))

## 7. Write `calibration.json`

In [ ]:
CAL = {"devices": names, "capabilities": [list(c) for c in caps],
       "torch": torch.__version__, "cuda": torch.version.cuda,
       "wandb_secret": SECRET,
       "micro_bench": STEP,
       "real_run": {"rounds": len(rounds), "config": cal_cfg,
                    "history": rounds, "wall_s": round(CAL_WALL, 1),
                    "prepack": prepack, "compile_modes": compile_modes,
                    "resident_GiB_after_load": resident,
                    "ms_per_step_batch512": round(MS_PER_STEP_512, 3),
                    "median_train_s": round(t_train, 1), "median_eval_s": round(t_eval, 1),
                    "median_commit_s": round(t_commit, 1),
                    "median_full_round_s": round(t_full, 1),
                    "ratio_batch256_over_512": ratio_256},
       "projection": PROJ}
with open("/kaggle/working/calibration.json", "w") as fh:
    json.dump(CAL, fh, indent=2)
print(json.dumps({k: v for k, v in CAL.items() if k != "real_run"}, indent=2)[:4000])
print("\nwrote /kaggle/working/calibration.json")

These are measurements of this session's hardware, not a guarantee about the next one.
Re-read live quota before launching a training run and set `MAX_HOURS` from the smaller
of remaining quota, the session limit, and `suggested_max_hours` above.

The 50- and 100-client projections scale the measured 20-client step cost; only the
20-client row is a direct measurement. The batch-256 ratio comes from the synthetic
micro-benchmark, so the 100-client projection carries that extra assumption.